In [ ]:
import os
import gc
import copy
import glob
import pickle
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler


# ============================================================
# 0. KIEN TRUC (MA TRAN) - thay the long-format 58 trieu dong
#   Panel : moi feature = 1 MA TRAN (n_series x n_days) -> sales 30.490 x 1.913
#   Lag   : dich cot (shift)              | Rolling: cumsum vectorized
#   Cap   : nhan voi MA TRAN NHOM THUA (sparse) -> L1/L2/L3/L9/L10
#   Multi-level: L1 (national) / L2 (state) / L3 (store) / L10 (item) + L12 (SKU base)
#   RAM uoc luong toan pipeline: ~4.5 GB (thay vi >20 GB long-format)
#   Zero-inflation: 15 PAST features (4 feature intermittency: days_since_last_sale,
#   nonzero_ratio_28/56, sales_rolling_mean_56) + log1p target + quantile loss;
#   WMAPE per-key dung nanmean (key tong actual = 0 -> NaN, KHONG gan 0).
# ============================================================

# ============================================================
# 1. ENVIRONMENT & GPU
# ============================================================

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

H, SEQ = 28, 56

# ------------------------------------------------------------
# FIXED TIME BOUNDARIES: single source of truth.
# Never derive split points from each series' max date.
#   VAL : encoder d_1802..d_1857 -> targets d_1858..d_1885
#   TEST: encoder d_1830..d_1885 -> targets d_1886..d_1913
# TRAIN targets start at d_57 because SEQ=56 (encoder d_1..d_56).
# ------------------------------------------------------------
START_DAY, END_DAY = 1, 1913
TRAIN_TARGET_START, TRAIN_TARGET_END = 57, 1857
VAL_START, VAL_END = 1858, 1885
TEST_START, TEST_END = 1886, 1913

# Preprocessing cutoffs MUST equal train-target end (leakage guard):
PRICE_STATS_END_DAY = TRAIN_TARGET_END   # price max/min/mean/std: train only
MEDIAN_FILL_END_DAY = TRAIN_TARGET_END   # dept price median fill: train only

PREPROCESS_VERSION = "v5_matrix_panel_log1p_d1_d1913_intermittency"
CONFIG_VERSION = "boundaries_d57_1857_val1858_1885_test1886_1913"

_boundary_errors = []
if H != 28 or SEQ != 56:
    _boundary_errors.append(f"SEQ/H must be 56/28, got {SEQ}/{H}")
if TRAIN_TARGET_START != SEQ + 1:
    _boundary_errors.append("TRAIN_TARGET_START must equal SEQ+1")
if VAL_START != TRAIN_TARGET_END + 1:
    _boundary_errors.append("VAL_START must equal TRAIN_TARGET_END+1")
if VAL_END - VAL_START + 1 != H:
    _boundary_errors.append("VAL window must have H days")
if TEST_START != VAL_END + 1:
    _boundary_errors.append("TEST_START must equal VAL_END+1")
if TEST_END - TEST_START + 1 != H:
    _boundary_errors.append("TEST window must have H days")
if TEST_END != END_DAY:
    _boundary_errors.append("TEST_END must equal END_DAY")
if PRICE_STATS_END_DAY > TRAIN_TARGET_END or MEDIAN_FILL_END_DAY > TRAIN_TARGET_END:
    _boundary_errors.append("preprocessing cutoff after train end (leak)")
if _boundary_errors:
    raise ValueError("Invalid boundary config: " + " | ".join(_boundary_errors))
del _boundary_errors

TRAIN_SAMPLE_N = None

# log1p target (zero-inflated retail sales)
LOG_TARGET = True

# capacity + epochs (OneCycle + early stopping on validation)
TFT_CFG = {
    "d_model": 64,
    "num_heads": 4,
    "epochs": 18,
    "batch_size": 512,
    "lr": 1e-3,
    "dropout": 0.1,
    "patience": 5,
}

# recency-weighted stride sampling (full d_1 history)
SAMPLING = {
    "recent_days": 300,
    "mid_days": 700,
    "recent_stride": 4,
    "mid_stride": 12,
    "old_stride": 48,
}

# DataLoader workers (0 = an toan; collate da vectorized theo batch nen khong can worker)
TFT_DATA_WORKERS = 0

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)

In [ ]:
# ============================================================
# 2. AUTOMATICALLY FIND M5 DATASET
# ============================================================

DATA_ROOT = "/kaggle/input"

required_files = [
    "calendar.csv",
    "sales_train_validation.csv",
    "sales_train_evaluation.csv",
    "sell_prices.csv",
]

DATA_DIR = None

# Search recursively for calendar.csv
calendar_matches = glob.glob(
    os.path.join(DATA_ROOT, "**", "calendar.csv"),
    recursive=True,
)

for calendar_path in calendar_matches:
    candidate_dir = os.path.dirname(calendar_path)

    # Verify that the directory contains the required files
    if all(
        os.path.isfile(os.path.join(candidate_dir, filename))
        for filename in required_files
    ):
        DATA_DIR = candidate_dir
        break


# If the complete dataset is not found, show available CSV files
if DATA_DIR is None:
    print("\nCSV files found under /kaggle/input:")

    for root, dirs, files in os.walk(DATA_ROOT):
        for filename in files:
            if filename.lower().endswith(".csv"):
                print(os.path.join(root, filename))

    raise FileNotFoundError(
        "\nCannot locate the complete M5 dataset. "
        "Please check the attached competition data."
    )


# ============================================================
# 3. OUTPUT DIRECTORY
# ============================================================

OUT_DIR = "/kaggle/working"

os.makedirs(OUT_DIR, exist_ok=True)

print("\nDATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)

print("\nDataset files:")
print(os.listdir(DATA_DIR))


# ============================================================
# 4. VERIFY DATA FILES
# ============================================================

for filename in required_files:
    filepath = os.path.join(DATA_DIR, filename)

    if not os.path.isfile(filepath):
        raise FileNotFoundError(
            f"Missing dataset file: {filepath}"
        )

    print(f"[OK] {filename}")


# ============================================================
# 5. LOAD M5 DATA
# ============================================================

calendar_path = os.path.join(DATA_DIR, "calendar.csv")
sales_validation_path = os.path.join(
    DATA_DIR, "sales_train_validation.csv"
)
sales_evaluation_path = os.path.join(
    DATA_DIR, "sales_train_evaluation.csv"
)
prices_path = os.path.join(DATA_DIR, "sell_prices.csv")

print("\nLoading calendar...")
calendar = pd.read_csv(calendar_path)

# P0 FIX RAM: KHONG nap full sales/prices (30.490 x 1.913 -> ~1GB ram thua).
# Pipeline ma tran tu doc lai file voi dtype int16; day chi xem header + mau.
print("Loading sales validation (preview only)...")
sales = pd.read_csv(sales_validation_path, nrows=5)

print("Loading sales evaluation (preview only)...")
sales_evaluation = pd.read_csv(sales_evaluation_path, nrows=5)

print("Loading sell prices (preview only)...")
sell_prices = pd.read_csv(prices_path, nrows=5)


# ============================================================
# 6. DATASET SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("M5 DATASET SUMMARY (preview - pipeline doc lai voi dtype int16)")
print("=" * 60)

print("Calendar shape:", calendar.shape)
print("Sales validation columns:", len(sales.columns), "| first day cols:", sales.columns.tolist()[6:11])
print("Sales evaluation columns:", len(sales_evaluation.columns))
print("Sell prices shape (preview):", sell_prices.shape)

print("\nCalendar columns:")
print(calendar.columns.tolist())

print("\nSales columns (first 15):")
print(sales.columns.tolist()[:15])

print("\nSell prices columns:")
print(sell_prices.columns.tolist())

print("\nSales validation preview:")
print(sales.iloc[:5, :10])

print("\nDataset initialization completed successfully.")
print("Device:", DEVICE)
print("Output directory:", OUT_DIR)

# Release eagerly-loaded frames; the PIPELINE cell re-reads with proper dtypes.
del sales, sales_evaluation, sell_prices, calendar
gc.collect()

In [ ]:
# UTILS: memory + metrics
def reduce_mem_usage(df, verbose=False):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        ct = df[col].dtypes
        if ct in numerics:
            cmin, cmax = df[col].min(), df[col].max()
            if str(ct)[:3] == 'int':
                if cmin > np.iinfo(np.int8).min and cmax < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif cmin > np.iinfo(np.int16).min and cmax < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif cmin > np.iinfo(np.int32).min and cmax < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if cmin > np.finfo(np.float32).min and cmax < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    if verbose:
        end = df.memory_usage().sum() / 1024**2
        print(f'mem {start:.1f}->{end:.1f} MB')
    return df

def _as_float(a):
    return np.asarray(a, dtype=np.float64).ravel()

def calc_rmse(a, b):
    a, b = _as_float(a), _as_float(b)
    return float(np.sqrt(np.mean((a - b) ** 2))) if a.size else float('nan')

def calc_mae(a, b):
    a, b = _as_float(a), _as_float(b)
    return float(np.mean(np.abs(a - b))) if a.size else float('nan')

def calc_mape(y_true, y_pred):
    a, p = _as_float(y_true), _as_float(y_pred)
    mask = (a != 0) & ~np.isnan(a)
    if not np.any(mask):
        return float('nan')
    return float(np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100.0)

def calc_wmape(a, b):
    a, b = _as_float(a), _as_float(b)
    den = np.abs(a).sum()
    return float('nan') if den == 0 else float(np.abs(a - b).sum() / den * 100)

def calc_bias(a, b):
    a, b = _as_float(a), _as_float(b)
    return float(np.mean(b - a)) if a.size else float('nan')

def calc_smape(a, b, eps=1e-8):
    a, b = _as_float(a), _as_float(b)
    den = (np.abs(a) + np.abs(b)) / 2.0 + eps
    return float(np.mean(np.abs(b - a) / den) * 100) if a.size else float('nan')

def calc_rmsse(y_true, y_pred, y_train):
    a, p, tr = _as_float(y_true), _as_float(y_pred), _as_float(y_train)
    nz = np.where(tr != 0)[0]
    if len(nz) == 0:
        scale_sq = 0.0
    else:
        tr_act = tr[nz[0]:]
        scale_sq = np.mean(np.diff(tr_act) ** 2) if len(tr_act) > 1 else 0.0
    rmse = float(np.sqrt(np.mean((a - p) ** 2))) if a.size else float('nan')
    if scale_sq <= 0 or np.isnan(scale_sq):
        return 0.0 if rmse == 0 else float('inf')
    return float(rmse / np.sqrt(scale_sq))

def forecast_metrics(y_true, y_pred, y_train=None, label=''):
    a, p = _as_float(y_true), _as_float(y_pred)
    if a.shape != p.shape:
        raise ValueError(f"{label}: shape mismatch {a.shape} vs {p.shape}")
    den = np.abs(a).sum()
    ratio = float(p.sum() / a.sum()) if a.sum() != 0 else float('nan')
    res = {
        'n_obs': int(a.size),
        'total_actual': float(a.sum()),
        'total_pred': float(p.sum()),
        'pred_actual_ratio': ratio,
        'WMAPE (%)': calc_wmape(a, p),
        'MAPE (%)': calc_mape(a, p),
        'MAE': calc_mae(a, p),
        'RMSE': calc_rmse(a, p),
        'Bias': calc_bias(a, p),
        'sMAPE (%)': calc_smape(a, p),
        'zero_actual_frac': float(np.mean(a == 0)) if a.size else float('nan'),
        'zero_denominator': bool(den == 0),
    }
    if y_train is not None:
        res['RMSSE'] = calc_rmsse(a, p, y_train)
    return res

def evaluate_forecast(y_true, y_pred, y_train=None):
    m = forecast_metrics(y_true, np.clip(np.asarray(y_pred, dtype=np.float64), 0, None), y_train=y_train)
    keys = ['RMSE', 'MAE', 'MAPE (%)', 'WMAPE (%)', 'sMAPE (%)']
    if y_train is not None:
        keys.append('RMSSE')
    return {k: m[k] for k in keys}

print('utils OK')


In [ ]:
# ============================================================
# PIPELINE MA TRAN (PANEL): L12 base -> L1/L2/L3/L10
# Khong bao gio "melt" 58.317.370 dong long-format.
# Moi feature = 1 MA TRAN (n_series x n_days); lag = dich cot;
# rolling = cumsum; tong hop cap = nhan voi ma tran nhom thua (sparse).
# ============================================================
import gc
import time
import warnings

import numpy as np
import pandas as pd
from scipy import sparse

T_PIPE = time.time()
N_DAYS = END_DAY - START_DAY + 1
if START_DAY != 1 or END_DAY != 1913:
    raise ValueError(f'Matrix pipeline expects START_DAY=1, END_DAY=1913, got {START_DAY}..{END_DAY}')

# ------------------------------------------------------------------
# 1. CALENDAR -> vector theo ngay (d_num -> index 0..N_DAYS-1)
# ------------------------------------------------------------------
cal = pd.read_csv(os.path.join(DATA_DIR, 'calendar.csv'))
cal['d_num'] = cal['d'].str[2:].astype(np.int32)
cal = cal[(cal['d_num'] >= START_DAY) & (cal['d_num'] <= END_DAY)].sort_values('d_num').reset_index(drop=True)
if len(cal) != N_DAYS:
    raise ValueError(f'Calendar thieu ngay: co {len(cal)}, can {N_DAYS}')

_wday = cal['wday'].to_numpy(np.int32)
_month = cal['month'].to_numpy(np.int32)
CAL_WM_YR_WK = cal['wm_yr_wk'].to_numpy(np.int32)
CAL_IS_WEEKEND = cal['wday'].isin([1, 2]).to_numpy(np.int8)
CAL_SIN_WDAY = np.sin(2 * np.pi * _wday / 7.0).astype(np.float32)
CAL_COS_WDAY = np.cos(2 * np.pi * _wday / 7.0).astype(np.float32)
CAL_SIN_MONTH = np.sin(2 * np.pi * _month / 12.0).astype(np.float32)
CAL_COS_MONTH = np.cos(2 * np.pi * _month / 12.0).astype(np.float32)

for _c in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
    if _c not in cal.columns:
        cal[_c] = 'None'
    cal[_c] = cal[_c].fillna('None')
CAL_HAS_EVENT = (cal['event_name_1'] != 'None').to_numpy(np.int8)
CAL_EVENT_COUNT = ((cal['event_name_1'] != 'None').to_numpy(np.int32)
                   + (cal['event_name_2'] != 'None').to_numpy(np.int32)).astype(np.int8)
CAL_EVT = {}
for _t in ['Cultural', 'Religious', 'Sporting', 'National']:
    CAL_EVT[_t] = (cal['event_type_1'] == _t).to_numpy(np.int8)

# days_to_next_event / days_since_last_event (vector, ngan nhu ban goc)
_n = len(cal)
_evt_idx = np.where(CAL_HAS_EVENT == 1)[0]
_d2n = np.full(_n, 999, dtype=np.int16)
_dsl = np.full(_n, 999, dtype=np.int16)
if len(_evt_idx) > 0:
    _nx = np.searchsorted(_evt_idx, np.arange(_n))
    _v = _nx < len(_evt_idx)
    _d2n[_v] = (_evt_idx[_nx[_v]] - np.arange(_n)[_v]).astype(np.int16)
    _pv = np.searchsorted(_evt_idx, np.arange(_n), side='right') - 1
    _vp = _pv >= 0
    _dsl[_vp] = (np.arange(_n)[_vp] - _evt_idx[_pv[_vp]]).astype(np.int16)
CAL_DAYS_TO_NEXT = np.clip(_d2n, 0, 60).astype(np.int16)
CAL_DAYS_SINCE = np.clip(_dsl, 0, 60).astype(np.int16)

CAL_SNAP = {}
for _st in ['CA', 'TX', 'WI']:
    _c = f'snap_{_st}'
    CAL_SNAP[_st] = cal[_c].to_numpy(np.int8) if _c in cal.columns else np.zeros(_n, np.int8)
del cal
gc.collect()

# ------------------------------------------------------------------
# 2. SALES (wide) -> MA TRAN (n_series x n_days) int16
# ------------------------------------------------------------------
_meta_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
_sales_path = os.path.join(DATA_DIR, 'sales_train_validation.csv')
_hdr = pd.read_csv(_sales_path, nrows=0)
_day_cols = [f'd_{i}' for i in range(START_DAY, END_DAY + 1)]
_missing = [c for c in _day_cols if c not in _hdr.columns]
if _missing:
    raise ValueError(f'sales_train_validation thieu ngay: {_missing[:5]}... ({len(_missing)} cols)')
del _hdr
_sales = pd.read_csv(_sales_path, usecols=_meta_cols + _day_cols,
                     dtype={c: np.int16 for c in _day_cols})
SALES = np.ascontiguousarray(_sales[_day_cols].to_numpy(np.int16))
N_SERIES = SALES.shape[0]
if SALES.shape[1] != N_DAYS:
    raise ValueError(f'Sales matrix shape {SALES.shape} != ({N_SERIES}, {N_DAYS})')
SERIES_ID = _sales['id'].astype(str).to_numpy()
META_ITEM = _sales['item_id'].astype(str).to_numpy()
META_DEPT = _sales['dept_id'].astype(str).to_numpy()
META_CAT = _sales['cat_id'].astype(str).to_numpy()
META_STORE = _sales['store_id'].astype(str).to_numpy()
META_STATE = _sales['state_id'].astype(str).to_numpy()
del _sales
gc.collect()
print(f'sales matrix: {SALES.shape[0]:,} series x {SALES.shape[1]:,} days '
      f'({SALES.nbytes / 1024**2:.0f} MB int16) | KHONG tao long-format 58M dong')

# code hoa metadata (global, khong phu thuoc chunk)
def _codes(arr):
    uniq = np.array(sorted(set(arr.tolist())))
    lut = {v: i + 1 for i, v in enumerate(uniq)}
    return uniq, np.array([lut[v] for v in arr], dtype=np.int64)

STORE_NAMES, STORE_CODES = _codes(META_STORE)
STATE_NAMES, STATE_CODES = _codes(META_STATE)
CAT_NAMES, CAT_CODES = _codes(META_CAT)
DEPT_NAMES, DEPT_CODES = _codes(META_DEPT)
ITEM_NAMES, ITEM_CODES = _codes(META_ITEM)
N_STORES, N_STATES, N_CATS, N_DEPTS, N_ITEMS = (len(STORE_NAMES), len(STATE_NAMES),
                                                len(CAT_NAMES), len(DEPT_NAMES), len(ITEM_NAMES))

# ------------------------------------------------------------------
# 3. PRICES -> ma tran (n_series x n_week), thong ke chi tren train
# ------------------------------------------------------------------
_weeks = np.unique(CAL_WM_YR_WK)
WEEKS = _weeks.astype(np.int32)
N_WEEKS = len(WEEKS)
_wk_col = np.searchsorted(WEEKS, CAL_WM_YR_WK)  # day -> week column
_wk_train_end = int(CAL_WM_YR_WK[PRICE_STATS_END_DAY - START_DAY])
_week_is_train = WEEKS <= _wk_train_end

_prices = pd.read_csv(os.path.join(DATA_DIR, 'sell_prices.csv'),
                      dtype={'sell_price': np.float32})
_price_wk = _prices['wm_yr_wk'].to_numpy(np.int32)
_keep = np.isin(_price_wk, WEEKS)
_price_wk = _price_wk[_keep]
_price_val = _prices['sell_price'].to_numpy(np.float32)[_keep]
_price_store = pd.Categorical(_prices['store_id'].astype(str),
                              categories=list(STORE_NAMES)).codes[_keep]
_price_item = pd.Categorical(_prices['item_id'].astype(str),
                             categories=list(ITEM_NAMES)).codes[_keep]
del _prices
gc.collect()
_price_wc = np.searchsorted(WEEKS, _price_wk)

# (item, store) -> series index
_pair = np.full((N_ITEMS, N_STORES), -1, dtype=np.int64)
_pair[ITEM_CODES - 1, STORE_CODES - 1] = np.arange(N_SERIES)
_price_sidx = _pair[_price_item, _price_store]
_ok = _price_sidx >= 0
PRICE_WEEK = np.full((N_SERIES, N_WEEKS), np.nan, dtype=np.float32)
PRICE_WEEK[_price_sidx[_ok], _price_wc[_ok]] = _price_val[_ok]
del _price_sidx, _price_val, _price_wk, _price_wc, _price_store, _price_item, _keep, _ok, _pair
gc.collect()

# thong ke gia chi tren tuan train (price_stats_end_wk)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _ptrain = PRICE_WEEK[:, _week_is_train]
    PRICE_MAX = np.nanmax(_ptrain, axis=1).astype(np.float32)
    PRICE_MEAN = np.nanmean(_ptrain, axis=1).astype(np.float32)
    del _ptrain
print(f'price week matrix: {PRICE_WEEK.shape[0]:,} x {PRICE_WEEK.shape[1]:,} | '
      f'train weeks <= {_wk_train_end}')

# discount (train-only price_max) + momentum (pct_change theo tuan co gia)
with np.errstate(divide='ignore', invalid='ignore'):
    _disc_week = np.where(np.isnan(PRICE_WEEK), np.nan, 1.0 - PRICE_WEEK / PRICE_MAX[:, None])
_disc_week = np.nan_to_num(_disc_week, nan=0.0, posinf=0.0, neginf=0.0)
np.clip(_disc_week, 0.0, 1.0, out=_disc_week)

_present = ~np.isnan(PRICE_WEEK)
_idx_present = np.where(_present, np.arange(N_WEEKS)[None, :], -1)
_prev = np.maximum.accumulate(_idx_present, axis=1)
_prev = np.roll(_prev, 1, axis=1)
_prev[:, 0] = -1
_prev_price = np.where(_prev >= 0,
                       PRICE_WEEK[np.arange(N_SERIES)[:, None], np.clip(_prev, 0, None)],
                       np.nan)
with np.errstate(divide='ignore', invalid='ignore'):
    _mom_week = PRICE_WEEK / _prev_price - 1.0
_mom_week = np.nan_to_num(_mom_week, nan=0.0, posinf=0.0, neginf=0.0)
del _present, _idx_present, _prev, _prev_price
gc.collect()

# broadcast tuan -> ngay
PRICE_DAY = np.ascontiguousarray(PRICE_WEEK[:, _wk_col])
DISC_DAY = np.ascontiguousarray(_disc_week[:, _wk_col])
MOM_DAY = np.ascontiguousarray(_mom_week[:, _wk_col])
del PRICE_WEEK, _disc_week, _mom_week
gc.collect()

# ------------------------------------------------------------------
# 4. MEDIAN FILL gia (chi tren ngay train) - giong ban goc
# ------------------------------------------------------------------
_n_train_days = MEDIAN_FILL_END_DAY - START_DAY + 1
PRICE_FILLED = PRICE_DAY.copy()
IS_LISTED = ~np.isnan(PRICE_DAY)
_dept_med = np.full(N_SERIES, np.nan, dtype=np.float64)
for _d in range(1, N_DEPTS + 1):
    _rows = np.where(DEPT_CODES == _d)[0]
    if len(_rows) == 0:
        continue
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _dept_med[_rows] = np.nanmedian(PRICE_DAY[_rows, :_n_train_days], axis=1)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _glob_med = float(np.nanmedian(PRICE_DAY[:, :_n_train_days]))
_nanmask = np.isnan(PRICE_FILLED)
if _nanmask.any():
    _rows, _cols = np.nonzero(_nanmask)
    _fill_vals = _dept_med[_rows]
    _bad = np.isnan(_fill_vals)
    _fill_vals[_bad] = _glob_med
    PRICE_FILLED[_rows, _cols] = _fill_vals
    del _rows, _cols, _fill_vals
del _nanmask
gc.collect()
print(f'median fill gia: dept={N_DEPTS} nhanh, global_median={_glob_med:.4f} (chi ngay train <= {MEDIAN_FILL_END_DAY})')

# ------------------------------------------------------------------
# 5. FEATURES L12 (ma tran) - cung semantics voi ban long-format
# ------------------------------------------------------------------
PAST_NAMES = [
    'sales', 'sales_lag_28', 'sales_lag_35', 'sales_rolling_mean_7', 'sales_rolling_mean_28',
    'sell_price_lag_28', 'price_discount_lag_28', 'price_momentum_lag_28',
    'sell_price_rolling_mean_7', 'is_listed', 'is_listed_lag_28',
    'days_since_last_sale', 'nonzero_ratio_28', 'nonzero_ratio_56', 'sales_rolling_mean_56',
]
FUT_NAMES = [
    'is_weekend', 'sin_wday', 'cos_wday', 'sin_month', 'cos_month', 'snap_active',
    'has_event', 'event_count', 'evt_Cultural', 'evt_Religious', 'evt_Sporting',
    'evt_National', 'days_to_next_event', 'days_since_last_event',
]
STATIC_NAMES = ['store_id', 'state_id', 'cat_id', 'dept_id', 'item_id']

PAST = np.zeros((len(PAST_NAMES), N_SERIES, N_DAYS), dtype=np.float32)


def _shift_into(dst, src, k):
    """dst[:, t] = src[:, t-k] (0 cho t < k) - giong pandas groupby.shift()."""
    dst[:, k:] = src[:, :-k]


def _roll_mean_shift_into(dst, src, w, shift=28):
    """mean cua src[t-shift-w+1 .. t-shift] (min_periods=1) - giong pandas rolling()."""
    S, D = src.shape
    chunk = 4000
    for lo in range(0, S, chunk):
        hi = min(S, lo + chunk)
        cs = np.zeros((hi - lo, D + 1), dtype=np.float64)
        np.cumsum(src[lo:hi], axis=1, dtype=np.float64, out=cs[:, 1:])
        t = np.arange(D)
        a = t - shift
        b = np.maximum(a - w + 1, 0)
        valid = a >= 0
        a_c = np.clip(a, 0, D)
        b_c = np.clip(b, 0, D)
        sums = cs[:, a_c + 1] - cs[:, b_c]
        counts = np.where(valid, a - b + 1, 0)
        out = dst[lo:hi]
        np.divide(sums, np.maximum(counts, 1)[None, :], out=out, where=valid[None, :])
        out[:, ~valid] = 0.0
        del cs, sums
    return dst


PAST[0] = SALES                                  # target (int16 -> float32)
_shift_into(PAST[1], SALES, 28)
_shift_into(PAST[2], SALES, 35)
_roll_mean_shift_into(PAST[3], SALES, 7, 28)
_roll_mean_shift_into(PAST[4], SALES, 28, 28)
_shift_into(PAST[5], PRICE_FILLED, 28)
_shift_into(PAST[6], DISC_DAY, 28)
_shift_into(PAST[7], MOM_DAY, 28)
_roll_mean_shift_into(PAST[8], PRICE_FILLED, 7, 28)
PAST[9] = IS_LISTED
_shift_into(PAST[10], IS_LISTED, 28)

# ---- 3b. FEATURE INTERMITTENCY (zero-inflation) ----
# Zero-inflated retail: phan lon so ngay/series co sales = 0. 4 feature nay
# cho model biet "phan bo zero" ma khong can doi loss/architecture:
#   days_since_last_sale : 0 = ban hom nay; chua tung ban -> t+1; cap 28
#   nonzero_ratio_28/56  : ty le ngay co sales>0 trong cua so ngay (shift=0)
#   sales_rolling_mean_56: muc yeu cau 56 ngay lien tuc (khong shift)
_t_arr = np.arange(N_DAYS, dtype=np.int32)
_nz = (PAST[0] > 0)
for _lo in range(0, N_SERIES, 4000):
    _hi = min(N_SERIES, _lo + 4000)
    _idx = np.where(_nz[_lo:_hi], _t_arr[None, :], np.int32(-1))
    _last = np.maximum.accumulate(_idx, axis=1)                 # index ban gan nhat (<=-1 neu chua ban)
    _dsls = np.where(_last >= 0, _t_arr[None, :] - _last, _t_arr[None, :] + 1)
    PAST[11, _lo:_hi] = np.clip(_dsls, 0, 28).astype(np.float32)
    del _idx, _last, _dsls
_roll_mean_shift_into(PAST[12], _nz, 28, 0)
_roll_mean_shift_into(PAST[13], _nz, 56, 0)
_roll_mean_shift_into(PAST[14], PAST[0], 56, 0)
del _nz, _t_arr

PRICE_MATRIX = PRICE_DAY.copy()
del PRICE_FILLED, PRICE_DAY, DISC_DAY, MOM_DAY, IS_LISTED, SALES
gc.collect()
print(f'PAST matrix: {PAST.shape[0]} features x {PAST.shape[1]:,} x {PAST.shape[2]:,} '
      f'({PAST.nbytes / 1024**2:.0f} MB float32)')

# ------------------------------------------------------------------
# 6. LAUNCH TRIM (chi giu tu ngay bat dau ban) -> start_idx
# ------------------------------------------------------------------
_active = (PAST[0] > 0) | (PAST[9] > 0)
_has_any = _active.any(axis=1)
START_IDX = np.argmax(_active, axis=1).astype(np.int32)
START_IDX[~_has_any] = -1
del _active
N_NO_LAUNCH = int((~_has_any).sum())
gc.collect()

# ------------------------------------------------------------------
# 7. FUT theo store (calendar + SNAP cua state) -> (n_store, n_days, n_fut)
# ------------------------------------------------------------------
_day_feats = np.stack([
    CAL_IS_WEEKEND, CAL_SIN_WDAY, CAL_COS_WDAY, CAL_SIN_MONTH, CAL_COS_MONTH,
    CAL_HAS_EVENT, CAL_EVENT_COUNT, CAL_EVT['Cultural'], CAL_EVT['Religious'],
    CAL_EVT['Sporting'], CAL_EVT['National'], CAL_DAYS_TO_NEXT, CAL_DAYS_SINCE,
], axis=1).astype(np.float32)   # 13 cot (thieu snap_active)
_store_state = {}
for _i in range(N_STORES):
    _st = np.unique(META_STATE[STORE_CODES == (_i + 1)])
    if len(_st) != 1:
        raise ValueError(f'Store {STORE_NAMES[_i]} thuoc nhieu state: {_st}')
    _store_state[_i] = str(_st[0])
FUT = np.empty((N_STORES, N_DAYS, len(FUT_NAMES)), dtype=np.float32)
for _i in range(N_STORES):
    _snap = CAL_SNAP[_store_state[_i]].astype(np.float32)
    FUT[_i, :, :5] = _day_feats[:, 0:5]
    FUT[_i, :, 5] = _snap
    FUT[_i, :, 6:] = _day_feats[:, 5:]
del _day_feats
gc.collect()

# ------------------------------------------------------------------
# 8. STATIC codes (1-based, 0 = unknown) + store index cho FUT
# ------------------------------------------------------------------
STATIC = np.zeros((N_SERIES, len(STATIC_NAMES)), dtype=np.int64)
STATIC[:, 0] = STORE_CODES
STATIC[:, 1] = STATE_CODES
STATIC[:, 2] = CAT_CODES
STATIC[:, 3] = DEPT_CODES
STATIC[:, 4] = ITEM_CODES
STATIC_VALUES = {
    'store_id': STORE_NAMES, 'state_id': STATE_NAMES, 'cat_id': CAT_NAMES,
    'dept_id': DEPT_NAMES, 'item_id': ITEM_NAMES,
}
SERIES_STORE_IDX = STORE_CODES - 1

# ------------------------------------------------------------------
# 9. MA TRAN NHOM THUA cho tung cap tong hop
# ------------------------------------------------------------------
def _group_matrix(codes, n_keys):
    rows = np.asarray(codes, dtype=np.int64) - 1
    cols = np.arange(len(rows), dtype=np.int64)
    return sparse.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                              shape=(n_keys, len(rows)))


_SD_CODE = (STORE_CODES * 1000 + DEPT_CODES)
_SD_UNIQ, _SD_INV = np.unique(_SD_CODE, return_inverse=True)
L9_KEYS = [f'{STORE_NAMES[s - 1]}|{DEPT_NAMES[d - 1]}'
           for (s, d) in sorted({(int(STORE_CODES[i]), int(DEPT_CODES[i])) for i in range(N_SERIES)})]

GROUPS = {
    'L1': (['ALL'], sparse.csr_matrix(np.ones((1, N_SERIES), np.float32))),
    'L2': (list(STATE_NAMES), _group_matrix(STATE_CODES, N_STATES)),
    'L3': (list(STORE_NAMES), _group_matrix(STORE_CODES, N_STORES)),
    'L9': (L9_KEYS, _group_matrix(_SD_INV + 1, len(_SD_UNIQ))),
    'L10': (list(ITEM_NAMES), _group_matrix(ITEM_CODES, N_ITEMS)),
}
del _SD_CODE, _SD_UNIQ, _SD_INV
gc.collect()

# ------------------------------------------------------------------
# 10. PANEL BUNDLE
# ------------------------------------------------------------------
class M5Panel:
    """Panel ma tran: moi feature = 1 ma tran (n_series x n_days)."""

    def __init__(self, past, past_names, fut, fut_names, static, static_names,
                 static_values, store_idx, start_idx, ids, item, dept, cat, store, state, groups, prices=None):
        self.past = past
        self.past_names = list(past_names)
        self.fut = fut
        self.fut_names = list(fut_names)
        self.static = static
        self.static_names = list(static_names)
        self.static_values = static_values
        self.store_idx = store_idx
        self.start_idx = start_idx
        self.ids = ids
        self.item = item
        self.dept = dept
        self.cat = cat
        self.store = store
        self.state = state
        self.groups = groups
        self.prices = prices
        self.n_series = past.shape[1]
        self.n_days = past.shape[2]

    def memory_mb(self):
        total = self.past.nbytes + self.fut.nbytes + self.static.nbytes
        return total / 1024 ** 2


PANEL = M5Panel(PAST, PAST_NAMES, FUT, FUT_NAMES, STATIC, STATIC_NAMES, STATIC_VALUES,
                SERIES_STORE_IDX, START_IDX, SERIES_ID, META_ITEM, META_DEPT, META_CAT,
                META_STORE, META_STATE, GROUPS, prices=PRICE_MATRIX)
_N_L9 = len(GROUPS['L9'][0])
del PAST, FUT, STATIC, SERIES_STORE_IDX, START_IDX, SERIES_ID, META_ITEM, META_DEPT, META_CAT, META_STORE, META_STATE, GROUPS
gc.collect()

print('=' * 70)
print(f'PANEL: series={PANEL.n_series:,} days={PANEL.n_days:,} | RAM ma tran ~{PANEL.memory_mb():,.0f} MB')
print(f'Levels: L1=1 L2={N_STATES} L3={N_STORES} L9={_N_L9} L10={N_ITEMS} L12={PANEL.n_series:,}')
print(f'Launch trim: series co du lieu={int((PANEL.start_idx >= 0).sum()):,} | '
      f'khong co du lieu={N_NO_LAUNCH:,}')
print(f'Pipeline ma tran xong trong {time.time() - T_PIPE:.1f}s (khong bao gio tao 58M dong)')
print('=' * 70)


In [ ]:
import copy, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# ---------- AMP tuong thich moi phien ban torch ----------
def _new_amp():
    try:
        return torch.amp.GradScaler('cuda'), True
    except Exception:
        return torch.cuda.amp.GradScaler(), False

def _autocast():
    try:
        return torch.amp.autocast('cuda')
    except Exception:
        return torch.cuda.amp.autocast()

# ---------- components (y het src/models/transformer_components.py) ----------
class GLU(nn.Module):
    def __init__(self, inp, out):
        super().__init__()
        self.fc_val = nn.Linear(inp, out); self.fc_gate = nn.Linear(inp, out)
    def forward(self, x):
        return self.fc_val(x) * torch.sigmoid(self.fc_gate(x))

class GRN(nn.Module):
    def __init__(self, inp, hid, out, dropout=0.1, context_dim=None):
        super().__init__()
        self.fc1 = nn.Linear(inp, hid)
        self.fc_context = nn.Linear(context_dim, hid, bias=False) if context_dim else None
        self.fc2 = nn.Linear(hid, hid); self.drop = nn.Dropout(dropout)
        self.glu = GLU(hid, out); self.norm = nn.LayerNorm(out)
        self.skip = nn.Linear(inp, out) if inp != out else nn.Identity()
    def forward(self, x, context=None):
        h = self.fc1(x)
        if context is not None and self.fc_context is not None:
            ctx = context.unsqueeze(1) if h.dim() == 3 else context
            h = h + self.fc_context(ctx)
        h = self.drop(self.fc2(F.elu(h)))
        return self.norm(self.skip(x) + self.glu(h))

class VSN(nn.Module):
    def __init__(self, n_vars, inp, hid, dropout=0.1, context_dim=None):
        super().__init__()
        self.var_grns = nn.ModuleList([GRN(inp, hid, hid, dropout) for _ in range(n_vars)])
        self.weight_grn = GRN(n_vars * inp, hid, n_vars, dropout, context_dim)
    def forward(self, flat, indiv, context=None):
        w = F.softmax(self.weight_grn(flat, context), dim=-1)
        stacked = torch.stack([g(v) for g, v in zip(self.var_grns, indiv)], dim=-2)
        return torch.sum(stacked * w.unsqueeze(-1), dim=-2), w

class InterpretableMHA(nn.Module):
    def __init__(self, heads, d_model, dropout=0.1):
        super().__init__()
        assert d_model % heads == 0
        self.h = heads; self.dk = d_model // heads
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.o = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        qh = self.q(q).view(B, -1, self.h, self.dk).transpose(1, 2)
        kh = self.k(k).view(B, -1, self.h, self.dk).transpose(1, 2)
        vs = self.v(v)
        sc = torch.matmul(qh, kh.transpose(-2, -1)) / (self.dk ** 0.5)
        if mask is not None:
            # FIX crash Half-overflow: -1e9 tran float16 -> dung min cua dtype
            sc = sc.masked_fill(mask == 0, torch.finfo(sc.dtype).min)
        aw = self.drop(F.softmax(sc, dim=-1))
        mean_aw = aw.mean(dim=1)
        return self.o(torch.matmul(mean_aw, vs)), mean_aw

class QuantileLoss(nn.Module):
    def __init__(self, quantiles=[0.1, 0.5, 0.9]):
        super().__init__(); self.q = quantiles
    def forward(self, pred, true):
        true = true.unsqueeze(-1); losses = []
        for i, q in enumerate(self.q):
            e = true - pred[..., i:i+1]
            losses.append(torch.max((q - 1) * e, q * e))
        return torch.mean(torch.cat(losses, dim=-1))

class StaticContext(nn.Module):
    def __init__(self, cards, d):
        super().__init__()
        self.embs = nn.ModuleDict({k: nn.Embedding(c + 1, d, padding_idx=0) for k, c in cards.items()})
    def forward(self, sx):
        return sum(self.embs[k](sx[k]) for k in self.embs)

class TFTGlobalNet(nn.Module):
    def __init__(self, npast, nfut, cards, d=16, heads=2, drop=0.1, seq=56, H=28, qs=[0.1, 0.5, 0.9]):
        super().__init__(); self.seq = seq
        self.ctx = StaticContext(cards, d)
        self.pe = nn.ModuleList([nn.Linear(1, d) for _ in range(npast)])
        self.fe = nn.ModuleList([nn.Linear(1, d) for _ in range(nfut)])
        self.pvsn = VSN(npast, d, d, drop, d); self.fvsn = VSN(nfut, d, d, drop, d)
        self.enc = nn.LSTM(d, d, batch_first=True); self.dec = nn.LSTM(d, d, batch_first=True)
        self.glu = GLU(d, d); self.norm = nn.LayerNorm(d)
        self.enrich = GRN(d, d, d, drop, d)
        self.attn = InterpretableMHA(heads, d, drop); self.post = GRN(d, d, d, drop)
        self.head = nn.Linear(d, len(qs))
    def forward(self, px, fx, sx):
        se = self.ctx(sx)
        pe = [e(px[:, :, i:i+1]) for i, e in enumerate(self.pe)]
        pv, _ = self.pvsn(torch.cat(pe, -1), pe, se)
        fe = [e(fx[:, :, i:i+1]) for i, e in enumerate(self.fe)]
        fv, _ = self.fvsn(torch.cat(fe, -1), fe, se)
        enc, (hn, cn) = self.enc(pv)
        dec, _ = self.dec(fv[:, self.seq:], (hn, cn))
        comb = torch.cat([enc, dec], 1); vsn = torch.cat([pv, fv[:, self.seq:]], 1)
        tf = self.norm(vsn + self.glu(comb)); en = self.enrich(tf, se)
        L = en.size(1)
        mask = torch.tril(torch.ones(L, L, dtype=torch.bool, device=en.device))
        ao, _ = self.attn(en, en, en, mask)
        out = self.post(en + ao)
        return self.head(out[:, self.seq:]), {}


# ---------- Dataset doc truc tiep tu MA TRAN (khong materialize long-format) ----------
# Window duoc chon theo ABSOLUTE target days [target_start, target_end].
# a = chi so ngay (0-based) bat dau encoder; targets = a+SEQ .. a+SEQ+H-1.
class MatrixTFTDataset(Dataset):
    def __init__(self, panel, eligible, seq=56, H=28, target_start=None, target_end=None,
                 p_mean=None, p_scale=None, f_mean=None, f_scale=None,
                 t_mean=0.0, t_scale=1.0, log_target=True,
                 recent_days=300, mid_days=700, recent_stride=4, mid_stride=12, old_stride=48):
        self.panel = panel
        self.seq = seq; self.H = H
        self.p_mean = p_mean; self.p_scale = p_scale
        self.f_mean = f_mean; self.f_scale = f_scale
        self.t_mean = t_mean; self.t_scale = t_scale
        self.log_target = log_target
        if target_start is None:
            target_start = int(globals()['TRAIN_TARGET_START'])
        if target_end is None:
            target_end = int(globals()['TRAIN_TARGET_END'])
        self.target_start = int(target_start)
        self.target_end = int(target_end)
        n_days = panel.n_days
        starts, sids = [], []
        empty_series = []
        for s in eligible:
            d0 = int(panel.start_idx[s])
            if d0 < 0:
                empty_series.append(int(s)); continue
            n_avail = self.target_end - d0      # so ngay tu launch den ngay target_end
            nw = max(0, n_avail - seq - H + 1)
            k = 0
            kept = 0
            while k < nw:
                tgt_lo = d0 + k + seq + 1     # d_num cua ngay target dau
                tgt_hi = tgt_lo + H - 1
                if tgt_lo < self.target_start or tgt_hi > self.target_end:
                    k += 1
                    continue
                age = self.target_end - tgt_hi
                if age < recent_days:
                    st = recent_stride
                elif age < recent_days + mid_days:
                    st = mid_stride
                else:
                    st = old_stride
                starts.append(d0 + k)
                sids.append(int(s))
                kept += 1
                k += st
            if kept == 0:
                empty_series.append(int(s))
        self.starts = np.asarray(starts, dtype=np.int64)
        self.sids = np.asarray(sids, dtype=np.int64)
        self.total = len(self.starts)
        self.empty_series = empty_series
        print(f'[sampling] windows/epoch={self.total:,} series={len(eligible):,} '
              f'empty={len(empty_series):,} (recent={recent_stride}, mid={mid_stride}, old={old_stride})')

    def __len__(self):
        return self.total

    def __getitem__(self, idx):
        # chi tra (series, ngay bat dau encoder) - viec trich feature lam vectorized trong collate
        return (int(self.sids[idx]), int(self.starts[idx]))


def _extract_windows(panel, s_arr, a_arr, seq, H, p_mean, p_scale,
                     f_mean, f_scale, t_mean, t_scale, log_target):
    """Trich window vectorized cho ca batch: p (B,L,P), f (B,L+H,F), t (B,H), st (B,K)."""
    s_arr = np.asarray(s_arr, dtype=np.int64)
    a_arr = np.asarray(a_arr, dtype=np.int64)
    B = len(s_arr)
    widx = a_arr[:, None] + np.arange(seq + H, dtype=np.int64)[None, :]
    p = panel.past[:, s_arr[:, None], widx[:, :seq]]          # (P, B, L)
    p = np.ascontiguousarray(p.transpose(1, 2, 0))              # (B, L, P)
    p = (p - p_mean[None, None, :]) / p_scale[None, None, :]
    f = panel.fut[panel.store_idx[s_arr][:, None], widx, :]    # (B, L+H, F)
    f = (f - f_mean[None, None, :]) / f_scale[None, None, :]
    t = panel.past[0, s_arr[:, None], widx[:, seq:seq + H]]    # (B, H)
    if log_target:
        t = np.log1p(t)
    t = (t - t_mean) / t_scale
    st = panel.static[s_arr]
    return p, f, t, st


def _make_collate(panel, seq, H, p_mean, p_scale, f_mean, f_scale, t_mean, t_scale, log_target):
    def collate(batch):
        s_arr = np.fromiter((b[0] for b in batch), dtype=np.int64, count=len(batch))
        a_arr = np.fromiter((b[1] for b in batch), dtype=np.int64, count=len(batch))
        p, f, t, st = _extract_windows(panel, s_arr, a_arr, seq, H, p_mean, p_scale,
                                       f_mean, f_scale, t_mean, t_scale, log_target)
        return (torch.from_numpy(p).float(), torch.from_numpy(f).float(),
                torch.from_numpy(st).long(), torch.from_numpy(t).float())
    return collate


def _peak_rss_mb():
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / 1024**2)
    except Exception:
        return float('nan')


class TFTGlobalForecaster:
    def __init__(self, past_cols, fut_cols, static_cols, target='sales', seq=56, H=28,
                 d=16, heads=2, lr=1e-3, batch=256, epochs=10, device=None,
                 log_target=True, dropout=0.1, patience=5, sampling=None):
        self.pc = past_cols; self.fc = fut_cols; self.sc = static_cols
        self.target = target; self.seq = seq; self.H = H
        self.d = d; self.heads = heads; self.lr = lr; self.batch = batch; self.epochs = epochs
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.log_target = log_target; self.dropout = dropout; self.patience = patience
        self.sampling = dict(sampling or {})
        self.ps = StandardScaler(); self.fs = StandardScaler(); self.ts = StandardScaler()
        self.maps = {}; self.cards = {}; self.model = None
        g = globals()
        self.train_target_start = int(g.get('TRAIN_TARGET_START', seq + 1))
        self.train_target_end = int(g.get('TRAIN_TARGET_END', g.get('END_DAY', 1913) - H))
        self.val_start = int(g.get('VAL_START', self.train_target_end + 1))
        self.val_end = int(g.get('VAL_END', self.val_start + H - 1))
        self.history = []
        self.best_epoch = None; self.best_val_loss = None
        self.stop_reason = None
        self.train_duration_s = None
        self.peak_rss_mb = None; self.peak_gpu_mb = None
        self.n_windows = None; self.val_series = 0; self.val_skipped = 0
        self.p_mean = None; self.p_scale = None
        self.f_mean = None; self.f_scale = None
        self.t_mean = 0.0; self.t_scale = 1.0

    def _y(self, vals):
        vals = np.asarray(vals, dtype=np.float64)
        return np.log1p(vals) if self.log_target else vals

    def _static_row(self, s, panel):
        out = np.zeros(len(self.sc), dtype=np.int64)
        for i, c in enumerate(self.sc):
            out[i] = int(panel.static[s, i])
        return out

    def _sync_scalers(self):
        """Lay mean/scale tu StandardScaler (giu tuong thich voi checkpoint cu)."""
        self.p_mean = np.asarray(self.ps.mean_, dtype=np.float32)
        self.p_scale = np.asarray(self.ps.scale_, dtype=np.float32)
        self.f_mean = np.asarray(self.fs.mean_, dtype=np.float32)
        self.f_scale = np.asarray(self.fs.scale_, dtype=np.float32)
        self.t_mean = np.float32(self.ts.mean_[0]); self.t_scale = np.float32(self.ts.scale_[0])

    def _set_scaler(self, scaler, mean, scale, n):
        scaler.mean_ = np.asarray(mean, dtype=np.float64)
        scaler.var_ = np.asarray(scale, dtype=np.float64) ** 2
        scaler.scale_ = np.asarray(scale, dtype=np.float64)
        scaler.n_samples_seen_ = int(n)
        scaler.n_features_in_ = len(np.atleast_1d(mean))

    # ---------- 1) thong ke scaler tren MA TRAN (chi vung train) ----------
    def _fit_scalers(self, panel, eligible):
        S, D = panel.n_series, panel.n_days
        train_end_idx = self.train_target_end - 1
        start = np.clip(panel.start_idx[eligible], 0, None).astype(np.int64)
        day = np.arange(D, dtype=np.int64)[None, :]
        valid = (day >= start[:, None]) & (day <= train_end_idx)
        cnt = int(valid.sum())
        if cnt == 0:
            raise ValueError('Khong co cell nao trong vung train.')

        def _stats(get_vals):
            s1 = 0.0; s2 = 0.0; n = 0
            for lo in range(0, len(eligible), 4000):
                sl = eligible[lo:lo + 4000]
                m = valid[lo:lo + 4000]
                v = get_vals(sl)
                v = np.where(m, v, 0.0)
                s1 += float(v.sum(dtype=np.float64))
                s2 += float((v * v).sum(dtype=np.float64))
                n += int(m.sum())
                del v, m
            mean = s1 / n
            var = max(s2 / n - mean * mean, 0.0)
            scale = float(np.sqrt(var))
            if not np.isfinite(scale) or scale < 1e-8:
                scale = 1.0
            return mean, scale, n

        p_mean = []; p_scale = []
        for i, name in enumerate(self.pc):
            m, sc, n = _stats(lambda sl, i=i: panel.past[i, sl, :].astype(np.float64))
            p_mean.append(m); p_scale.append(sc)
        p_mean = np.asarray(p_mean); p_scale = np.asarray(p_scale)
        self._set_scaler(self.ps, p_mean, p_scale, cnt)

        # target: log1p(sales) tren cung mask
        tm, ts_, _ = _stats(lambda sl: np.log1p(panel.past[0, sl, :].astype(np.float64)))
        self._set_scaler(self.ts, [tm], [ts_], cnt)

        # FUT: weight = so series dang active cua store trong tung ngay
        f_sum = np.zeros(len(self.fc), dtype=np.float64)
        f_sq = np.zeros(len(self.fc), dtype=np.float64)
        w_tot = 0.0
        store_of = panel.store_idx[eligible]
        for st_i in range(panel.fut.shape[0]):
            rows = np.where(store_of == st_i)[0]
            if len(rows) == 0:
                continue
            starts = np.sort(panel.start_idx[eligible[rows]].astype(np.int64))
            starts = starts[starts >= 0]
            if len(starts) == 0:
                continue
            w = np.searchsorted(starts, np.arange(D, dtype=np.int64), side='right').astype(np.float64)
            w[train_end_idx + 1:] = 0.0
            x = panel.fut[st_i, :train_end_idx + 1, :].astype(np.float64)
            ww = w[:train_end_idx + 1][:, None]
            f_sum += (x * ww).sum(axis=0)
            f_sq += (x * x * ww).sum(axis=0)
            w_tot += float(ww.sum())
        f_mean = f_sum / max(w_tot, 1.0)
        f_var = np.maximum(f_sq / max(w_tot, 1.0) - f_mean ** 2, 0.0)
        f_scale = np.sqrt(f_var)
        f_scale[~np.isfinite(f_scale) | (f_scale < 1e-8)] = 1.0
        self._set_scaler(self.fs, f_mean, f_scale, int(w_tot))
        self._sync_scalers()
        print(f'[scaler] past cells={cnt:,} | target mean={tm:.4f} scale={ts_:.4f} | fut weight={w_tot:,.0f}')
        del valid, day

    # ---------- 2) validation theo batch (khong gom toan bo series) ----------
    @torch.no_grad()
    def _val_eval(self, panel, eligible, crit):
        self.model.eval()
        L, H = self.seq, self.H
        a = self.train_target_end - self.seq          # encoder d_1802..d_1857 (0-based)
        bs = 1024
        loss_tot = 0.0; n_tot = 0
        preds = []; ys = []
        for lo in range(0, len(eligible), bs):
            sl = eligible[lo:lo + bs]
            s_arr = sl
            a_arr = np.full(len(sl), a, dtype=np.int64)
            p, f, y, st = _extract_windows(panel, s_arr, a_arr, L, H,
                                           self.p_mean, self.p_scale, self.f_mean, self.f_scale,
                                           self.t_mean, self.t_scale, self.log_target)
            pt = torch.tensor(p, dtype=torch.float32, device=self.device)
            ft = torch.tensor(f, dtype=torch.float32, device=self.device)
            sd = {c: torch.tensor(st[:, j], dtype=torch.long, device=self.device)
                  for j, c in enumerate(self.sc)}
            yt = torch.tensor(y, dtype=torch.float32, device=self.device)
            q, _ = self.model(pt, ft, sd)
            loss_tot += float(crit(q, yt).item()) * len(sl)
            n_tot += len(sl)
            med = q[:, :, 1].detach().cpu().numpy().astype(np.float64)
            v = self.ts.inverse_transform(med.reshape(-1, 1)).reshape(med.shape)
            if self.log_target:
                v = np.expm1(v)
            preds.append(np.clip(v, 0, None))
            ys.append(panel.past[0, sl, a + L:a + L + H].astype(np.float64))
            del p, f, y, st, pt, ft, sd, yt, q
        loss = loss_tot / max(n_tot, 1)
        P = np.concatenate(preds, axis=0) if preds else np.zeros((0, H))
        Y = np.concatenate(ys, axis=0) if ys else np.zeros((0, H))
        if P.size:
            den = np.abs(Y).sum()
            wmape = float('nan') if den == 0 else float(np.abs(P - Y).sum() / den * 100)
            mae = float(np.abs(P - Y).mean())
        else:
            wmape = float('nan'); mae = float('nan')
        self.model.train()
        return loss, n_tot, wmape, mae

    def fit(self, panel, eligible):
        import time
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)
        eligible = np.asarray(eligible, dtype=np.int64)
        if len(eligible) == 0:
            raise ValueError('Khong co series nao de train.')
        print(f'[TFT] train {len(eligible):,} series on {self.device} | log_target={self.log_target} | '
              f'targets d_{self.train_target_start}..d_{self.train_target_end}')
        t_start = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        # 1) categorical maps: codes GLOBAL 1..N (panel.static) -> dung chung khi predict
        for j, c in enumerate(self.sc):
            names = getattr(panel, 'static_values', {}).get(c) if hasattr(panel, 'static_values') else None
            if names is not None:
                self.maps[c] = {str(names[k]): k + 1 for k in range(len(names))}
                self.cards[c] = len(names)
            else:
                u = sorted({int(v) for v in np.unique(panel.static[eligible, j]) if v > 0})
                self.maps[c] = {str(v): i + 1 for i, v in enumerate(u)}
                self.cards[c] = len(u)

        # 2) scalers tren ma tran
        self._fit_scalers(panel, eligible)

        # 3) dataset doc truc tiep tu ma tran
        ds = MatrixTFTDataset(panel, eligible,
                               seq=self.seq, H=self.H,
                               target_start=self.train_target_start,
                               target_end=self.train_target_end,
                               p_mean=self.p_mean, p_scale=self.p_scale,
                               f_mean=self.f_mean, f_scale=self.f_scale,
                               t_mean=self.t_mean, t_scale=self.t_scale,
                               log_target=self.log_target, **self.sampling)
        self.n_windows = len(ds)
        if self.n_windows == 0:
            raise ValueError('0 training windows: check boundaries / target_start / target_end.')
        _n_workers = int(globals().get('TFT_DATA_WORKERS', 0) or 0)
        _collate = _make_collate(panel, self.seq, self.H, self.p_mean, self.p_scale,
                                 self.f_mean, self.f_scale, self.t_mean, self.t_scale,
                                 self.log_target)
        try:
            loader = DataLoader(ds, batch_size=self.batch, shuffle=True, num_workers=_n_workers,
                                collate_fn=_collate, pin_memory=(self.device == 'cuda'))
        except Exception:
            loader = DataLoader(ds, batch_size=self.batch, shuffle=True, num_workers=0,
                                collate_fn=_collate, pin_memory=(self.device == 'cuda'))
        self.model = TFTGlobalNet(len(self.pc), len(self.fc), self.cards,
                                  self.d, self.heads, drop=self.dropout,
                                  seq=self.seq, H=self.H).to(self.device)
        crit = QuantileLoss()
        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=1e-5)
        steps = len(ds) // self.batch + (1 if len(ds) % self.batch else 0)
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=self.lr, total_steps=max(1, steps * self.epochs), pct_start=0.3)
        scaler_amp, _ = _new_amp() if self.device == 'cuda' else (None, True)

        # 4) validation tai bien co dinh
        best, best_state, bad = float('inf'), None, 0
        for ep in range(1, self.epochs + 1):
            self.model.train(); tot = 0.0
            for px, fx, sx, y in loader:
                px, fx, y = px.to(self.device), fx.to(self.device), y.to(self.device)
                sd = {c: sx[:, i].to(self.device) for i, c in enumerate(self.sc)}
                opt.zero_grad()
                if self.device == 'cuda':
                    with _autocast():
                        qp, _ = self.model(px, fx, sd); loss = crit(qp, y)
                    scaler_amp.scale(loss).backward(); scaler_amp.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    scaler_amp.step(opt); scaler_amp.update()
                else:
                    qp, _ = self.model(px, fx, sd); loss = crit(qp, y); loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0); opt.step()
                sched.step()
                tot += loss.item() * len(px)
            tot /= max(1, len(ds))
            vl, n_val, vw, vm = self._val_eval(panel, eligible, crit)
            lr_now = float(opt.param_groups[0]['lr'])
            print(f'[TFT] ep {ep}/{self.epochs} train={tot:.5f} val={vl:.5f} '
                  f'val_WMAPE={vw:.2f}% val_MAE={vm:.3f} val_series={n_val:,} lr={lr_now:.2e}', flush=True)
            self.history.append({'epoch': ep, 'train_loss': float(tot), 'val_loss': float(vl),
                                 'val_wmape': float(vw), 'val_mae': float(vm), 'lr': lr_now})
            if vl < best - 1e-6:
                best = vl; best_state = copy.deepcopy(self.model.state_dict()); bad = 0
                self.best_epoch = ep; self.best_val_loss = float(vl)
            else:
                bad += 1
            if bad >= self.patience:
                self.stop_reason = f'early_stop_epoch_{ep}_patience_{self.patience}'
                print(f'[TFT] early stop @ ep {ep} (best_val={best:.5f} @ ep {self.best_epoch})')
                break
        else:
            self.stop_reason = f'max_epochs_{self.epochs}'
        if best_state is not None:
            self.model.load_state_dict(best_state, strict=True)
        self.val_series = int(n_val)
        self.train_duration_s = float(time.time() - t_start)
        self.peak_rss_mb = _peak_rss_mb()
        self.peak_gpu_mb = float(torch.cuda.max_memory_allocated() / 1024**2) if torch.cuda.is_available() else 0.0
        print(f'[TFT] done | best_val={best:.5f} @ ep {self.best_epoch}/{self.epochs} | '
              f'stop={self.stop_reason} | {self.train_duration_s/60:.1f} min | '
              f'peak_RSS={self.peak_rss_mb:.0f} MB | peak_GPU={self.peak_gpu_mb:.0f} MB | '
              f'windows/epoch={self.n_windows:,}')
        return self

    @torch.no_grad()
    def predict_panel(self, panel, sidx, a):
        """Du bao H ngay cho nhieu series (a = chi so ngay 0-based bat dau encoder)."""
        self.model.eval()
        L, H = self.seq, self.H
        sidx = np.asarray(sidx, dtype=np.int64)
        a_arr = np.full(len(sidx), int(a), dtype=np.int64)
        p, f, _, st = _extract_windows(panel, sidx, a_arr, L, H,
                                       self.p_mean, self.p_scale, self.f_mean, self.f_scale,
                                       self.t_mean, self.t_scale, self.log_target)
        pt = torch.tensor(p, dtype=torch.float32, device=self.device)
        ft = torch.tensor(f, dtype=torch.float32, device=self.device)
        sd = {c: torch.tensor(st[:, j], dtype=torch.long, device=self.device)
              for j, c in enumerate(self.sc)}
        q, _ = self.model(pt, ft, sd)
        med = q[:, :, 1].detach().cpu().numpy().astype(np.float64)
        v = self.ts.inverse_transform(med.reshape(-1, 1)).reshape(med.shape)
        if self.log_target:
            v = np.expm1(v)
        v = np.clip(v, 0, None)
        if not np.isfinite(v).all():
            raise FloatingPointError('Non-finite prediction.')
        return v

    def save(self, path):
        import pickle
        torch.save(self.model.state_dict(), path + '.weights.pt')
        meta = {
            'maps': self.maps, 'cards': self.cards,
            'ps': self.ps, 'fs': self.fs, 'ts': self.ts,
            'pc': self.pc, 'fc': self.fc, 'sc': self.sc,
            'target': self.target, 'seq': self.seq, 'H': self.H,
            'd': self.d, 'heads': self.heads,
            'log_target': self.log_target, 'dropout': self.dropout,
            'patience': self.patience, 'sampling': self.sampling,
            'lr': self.lr, 'batch': self.batch, 'epochs': self.epochs,
            'history': self.history,
            'best_epoch': self.best_epoch, 'best_val_loss': self.best_val_loss,
            'stop_reason': self.stop_reason,
            'train_duration_s': self.train_duration_s,
            'peak_rss_mb': self.peak_rss_mb, 'peak_gpu_mb': self.peak_gpu_mb,
            'n_windows': self.n_windows,
            'val_series': self.val_series, 'val_skipped': self.val_skipped,
            'boundaries': {
                'train_target_start': self.train_target_start,
                'train_target_end': self.train_target_end,
                'val_start': self.val_start, 'val_end': self.val_end,
                'test_start': int(globals().get('TEST_START', self.val_end + 1)),
                'test_end': int(globals().get('TEST_END', self.val_end + self.H)),
            },
            'preprocess_version': globals().get('PREPROCESS_VERSION', 'unknown'),
            'config_version': globals().get('CONFIG_VERSION', 'unknown'),
            'torch_version': torch.__version__,
            'feature_schema': {'past': list(self.pc), 'fut': list(self.fc), 'static': list(self.sc)},
            'target_transform': 'log1p' if self.log_target else 'none',
            'quantiles': [0.1, 0.5, 0.9],
            'panel_representation': 'matrix (n_series x n_days), khong dung long-format 58M dong',
        }
        with open(path + '.meta.pkl', 'wb') as f:
            pickle.dump(meta, f)
        print('saved', path + '.weights.pt + .meta.pkl')


print('TFT model (matrix panel + lazy windows + real validation) OK')

In [ ]:
import gc, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ---- 0. cau hinh chot: FULL, khong sample ----
TRAIN_SAMPLE_N = None
print(f'TRAIN_SAMPLE_N = {TRAIN_SAMPLE_N} (None = full {PANEL.n_series:,} series)')

# ---- 1. FEATURE CONFIG (lay truc tiep tu PANEL, phai khop pipeline) ----
PAST = list(PANEL.past_names)
FUT = list(PANEL.fut_names)
STATIC = list(PANEL.static_names)
_EXPECT_PAST = ['sales', 'sales_lag_28', 'sales_lag_35', 'sales_rolling_mean_7', 'sales_rolling_mean_28',
                'sell_price_lag_28', 'price_discount_lag_28', 'price_momentum_lag_28',
                'sell_price_rolling_mean_7', 'is_listed', 'is_listed_lag_28',
                'days_since_last_sale', 'nonzero_ratio_28', 'nonzero_ratio_56', 'sales_rolling_mean_56']
_EXPECT_FUT = ['is_weekend', 'sin_wday', 'cos_wday', 'sin_month', 'cos_month', 'snap_active',
               'has_event', 'event_count', 'evt_Cultural', 'evt_Religious', 'evt_Sporting',
               'evt_National', 'days_to_next_event', 'days_since_last_event']
_EXPECT_STATIC = ['store_id', 'state_id', 'cat_id', 'dept_id', 'item_id']
if PAST != _EXPECT_PAST or FUT != _EXPECT_FUT or STATIC != _EXPECT_STATIC:
    raise ValueError('Schema feature panel khong khop mong doi (PAST/FUT/STATIC).')
print(f'feature schema OK | PAST={len(PAST)} FUT={len(FUT)} STATIC={len(STATIC)}')

# ---- 2. CONFIG CHECK ----
for _need in ['PANEL', 'SEQ', 'H', 'OUT_DIR', 'LOG_TARGET', 'TRAIN_TARGET_START', 'TRAIN_TARGET_END',
              'VAL_START', 'VAL_END', 'TEST_START', 'TEST_END', 'PREPROCESS_VERSION', 'CONFIG_VERSION']:
    if _need not in globals():
        raise NameError(f"Thieu '{_need}': chay cell CONFIG + PIPELINE truoc.")
for _need in ['TFTGlobalForecaster', 'TFTGlobalNet']:
    if _need not in globals():
        raise NameError(f"Thieu '{_need}': chay cell DINH NGHIA MODEL truoc.")

# ---- 3. ELIGIBLE SERIES (fixed boundaries, khong dung max date tung series) ----
_train_end_idx = TRAIN_TARGET_END - 1
_val_start_idx = VAL_START - 1
_test_start_idx = TEST_START - 1
_a_min = TRAIN_TARGET_START - SEQ - 1     # encoder bat dau toi da
_a_max = TRAIN_TARGET_END - SEQ - H       # encoder cuoi cung (target vua het bien train)
_start = PANEL.start_idx
excluded = {'no_launch': 0, 'short_train': 0, 'no_train_window': 0,
            'bad_val_days': 0, 'bad_test_days': 0}
_eligible = []
for _s in range(PANEL.n_series):
    _d0 = int(_start[_s])
    if _d0 < 0:
        excluded['no_launch'] += 1; continue
    if (_train_end_idx - _d0 + 1) < SEQ:
        excluded['short_train'] += 1; continue
    if _d0 > _a_max:
        excluded['no_train_window'] += 1; continue
    if _d0 > _val_start_idx:
        excluded['bad_val_days'] += 1; continue
    if _d0 > _test_start_idx:
        excluded['bad_test_days'] += 1; continue
    _eligible.append(_s)
eligible = np.asarray(_eligible, dtype=np.int64)
print(f'Eligible series: {len(eligible):,}/{PANEL.n_series:,} | excluded: {excluded}')

# ---- 4. SUBSAMPLE (optional, stratified theo cat x store) ----
if TRAIN_SAMPLE_N is not None and len(eligible) > TRAIN_SAMPLE_N:
    _meta = pd.DataFrame({
        's': eligible,
        'cat': PANEL.cat[eligible],
        'store': PANEL.store[eligible],
    })
    _picked = []
    for _, grp in _meta.groupby(['cat', 'store'], sort=True):
        _k = max(1, int(len(grp) * TRAIN_SAMPLE_N / len(_meta)))
        _picked += grp.sample(n=min(_k, len(grp)), random_state=SEED)['s'].tolist()
    eligible = np.asarray(sorted(set(_picked))[:TRAIN_SAMPLE_N], dtype=np.int64)
    print(f'sample train {len(eligible):,}/{PANEL.n_series:,}')
else:
    print(f'train full {len(eligible):,} series')
if len(eligible) == 0:
    raise ValueError('Khong co series nao du dieu kien train.')

# ---- 5. SPLIT AUDIT (fixed boundaries) ----
SPLIT_AUDIT = {
    'n_train': int(len(eligible)), 'n_val': int(len(eligible)), 'n_test': int(len(eligible)),
    'train_days_min': int(TRAIN_TARGET_START), 'train_days_max': int(TRAIN_TARGET_END),
    'no_val_test_rows_in_train': True,
    'val_days': (int(VAL_START), int(VAL_END)),
    'test_days': (int(TEST_START), int(TEST_END)),
    'excluded': dict(excluded),
    'expected_series': int(PANEL.n_series),
    'encoder_window': (int(_a_min), int(_a_max)),
    'representation': 'matrix panel (n_series x n_days)',
}
coverage_train = pd.DataFrame({
    'expected_series': [PANEL.n_series],
    'built_series': [len(eligible)],
    **{k: [v] for k, v in excluded.items()},
    'train_target_start': [TRAIN_TARGET_START], 'train_target_end': [TRAIN_TARGET_END],
    'val_start': [VAL_START], 'val_end': [VAL_END],
    'test_start': [TEST_START], 'test_end': [TEST_END],
})
coverage_train.to_csv(os.path.join(OUT_DIR, 'coverage_train.csv'), index=False)
print(coverage_train.to_string(index=False))
del _eligible
gc.collect()

# ---- 6. FIT (doc truc tiep tu ma tran; validation fixed d_1858..d_1885) ----
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass
print("\n--- Training Global TFT (log1p + real validation + matrix panel) ---")
tft = TFTGlobalForecaster(
    PAST, FUT, STATIC,
    target='sales',
    seq=SEQ, H=H,
    d=TFT_CFG['d_model'],
    heads=TFT_CFG['num_heads'],
    lr=TFT_CFG['lr'],
    batch=TFT_CFG['batch_size'],
    epochs=TFT_CFG['epochs'],
    log_target=LOG_TARGET,
    dropout=TFT_CFG.get('dropout', 0.1),
    patience=TFT_CFG.get('patience', 5),
    sampling=SAMPLING,
).fit(PANEL, eligible)
print("\nTFT training completed.")
print(f"Training series: {len(eligible):,} | SEQ={SEQ} H={H} | log_target={LOG_TARGET}")
print(f"Best epoch: {tft.best_epoch}/{tft.epochs} | best val loss: {tft.best_val_loss:.5f} "
      f"| stop: {tft.stop_reason}")

# ---- 7. SAVE CHECKPOINT + TRAINING LOG ----
import json
tft.save(os.path.join(OUT_DIR, 'tft_global_full30490'))
pd.DataFrame(tft.history).to_csv(os.path.join(OUT_DIR, 'tft_training_log.csv'), index=False)
with open(os.path.join(OUT_DIR, 'tft_training_config.json'), 'w', encoding='utf-8') as _f:
    json.dump({
        'boundaries': {
            'train_target_start': TRAIN_TARGET_START, 'train_target_end': TRAIN_TARGET_END,
            'val_start': VAL_START, 'val_end': VAL_END,
            'test_start': TEST_START, 'test_end': TEST_END,
            'price_stats_end': PRICE_STATS_END_DAY, 'median_fill_end': MEDIAN_FILL_END_DAY,
        },
        'tft_cfg': TFT_CFG, 'sampling': SAMPLING, 'seq': SEQ, 'H': H,
        'log_target': LOG_TARGET, 'preprocess_version': PREPROCESS_VERSION,
        'config_version': CONFIG_VERSION, 'torch_version': torch.__version__,
        'n_train_series': int(len(eligible)), 'n_windows_per_epoch': tft.n_windows,
        'best_epoch': tft.best_epoch, 'best_val_loss': tft.best_val_loss,
        'stop_reason': tft.stop_reason, 'train_duration_s': tft.train_duration_s,
        'peak_rss_mb': tft.peak_rss_mb, 'peak_gpu_mb': tft.peak_gpu_mb,
        'panel_ram_mb': round(PANEL.memory_mb(), 1),
        'excluded': excluded,
    }, _f, indent=2, ensure_ascii=False, default=str)
print('saved training log + config')
del eligible
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass
print("Checkpoint saved + RAM released. Chay tiep cell PREDICT.")

In [ ]:
import gc
import glob
import hashlib
import json
import os
import pickle
import shutil
import numpy as np
import pandas as pd
import torch

# ---- 0. guards ----
for _need in ['PANEL', 'SEQ', 'H', 'OUT_DIR', 'LOG_TARGET', 'TRAIN_TARGET_START', 'TRAIN_TARGET_END',
              'VAL_START', 'VAL_END', 'TEST_START', 'TEST_END', 'PREPROCESS_VERSION', 'CONFIG_VERSION']:
    if _need not in globals():
        raise NameError(f"Thieu '{_need}': chay cell CONFIG + PIPELINE truoc.")
for _need in ['TFTGlobalForecaster', 'TFTGlobalNet']:
    if _need not in globals():
        raise NameError(f"Thieu '{_need}': chay cell DINH NGHIA MODEL truoc.")

# ---- 1. checkpoint input (dataset attach) ----
_hits = glob.glob('/kaggle/input/**/tft_global_full30490.weights.pt', recursive=True)
CKPT_IN = os.path.dirname(_hits[0]) if _hits else OUT_DIR
print('CKPT_IN =', CKPT_IN)
print('OUT_DIR =', repr(OUT_DIR))

_EXPECTED_BOUNDARIES = {
    'train_target_start': int(TRAIN_TARGET_START), 'train_target_end': int(TRAIN_TARGET_END),
    'val_start': int(VAL_START), 'val_end': int(VAL_END),
    'test_start': int(TEST_START), 'test_end': int(TEST_END),
}

# ---- 2. load model from checkpoint if kernel restarted (STRICT keys) ----
if 'tft' not in globals():
    _w = os.path.join(CKPT_IN, 'tft_global_full30490.weights.pt')
    _m = os.path.join(CKPT_IN, 'tft_global_full30490.meta.pkl')
    if not (os.path.isfile(_w) and os.path.isfile(_m)):
        raise NameError(
            f"Chua co model (OUT_DIR={OUT_DIR!r}, CKPT_IN={CKPT_IN!r}): "
            "kiem tra dataset 'tft m5 checkpoint' da Add Input chua.")
    with open(_m, 'rb') as f:
        _meta = pickle.load(f)
    if _meta['seq'] != SEQ or _meta['H'] != H or _meta.get('target') != 'sales':
        raise ValueError(
            f"Checkpoint (seq={_meta['seq']}, H={_meta['H']}, target={_meta.get('target')!r}) "
            f"khac phien hien tai (seq={SEQ}, H={H}, target='sales') -> dung config hoac train lai.")
    if _meta.get('log_target', False) != LOG_TARGET:
        raise ValueError(
            f"Checkpoint log_target={_meta.get('log_target', False)} != LOG_TARGET={LOG_TARGET}. "
            "Checkpoint cu (raw target) -> train lai / attach checkpoint moi.")
    _b = _meta.get('boundaries')
    if not _b:
        raise ValueError('Checkpoint thieu boundaries -> khong the xac minh split; hay train lai.')
    _bad_b = {k: (_b.get(k), v) for k, v in _EXPECTED_BOUNDARIES.items() if _b.get(k) != v}
    if _bad_b:
        raise ValueError(f'Checkpoint boundaries khac config hien tai: {_bad_b}')
    tft = TFTGlobalForecaster(_meta['pc'], _meta['fc'], _meta['sc'],
        target=_meta['target'], seq=_meta['seq'], H=_meta['H'],
        d=_meta['d'], heads=_meta['heads'],
        log_target=_meta['log_target'],
        dropout=_meta.get('dropout', 0.1),
        patience=_meta.get('patience', 5),
        sampling=_meta.get('sampling', {}))
    tft.maps, tft.cards = _meta['maps'], _meta['cards']
    tft.ps, tft.fs, tft.ts = _meta['ps'], _meta['fs'], _meta['ts']
    tft.history = _meta.get('history', [])
    tft.best_epoch = _meta.get('best_epoch')
    tft.best_val_loss = _meta.get('best_val_loss')
    tft.model = TFTGlobalNet(len(tft.pc), len(tft.fc), tft.cards,
                             tft.d, tft.heads, drop=tft.dropout,
                             seq=tft.seq, H=tft.H).to(tft.device)
    _sd = torch.load(_w, map_location='cpu')
    _mk = set(tft.model.state_dict().keys())
    _fixed, _n_remap = {}, 0
    for k, v in _sd.items():
        if k in _mk:
            _fixed[k] = v
        elif k.startswith('ctx.') and ('ctx.embs.' + k[4:]) in _mk:
            _fixed['ctx.embs.' + k[4:]] = v; _n_remap += 1
        elif k.startswith('ctx.embs.') and ('ctx.' + k[9:]) in _mk:
            _fixed['ctx.' + k[9:]] = v; _n_remap += 1
        else:
            _fixed[k] = v
    _missing = sorted(_mk - set(_fixed))
    _unexpected = sorted(set(_fixed) - _mk)
    if _missing or _unexpected:
        raise ValueError(
            f'Checkpoint khop 100%: missing={_missing[:10]} unexpected={_unexpected[:10]} '
            '(khong dung strict=False de che giau loi kien truc).')
    tft.model.load_state_dict(_fixed, strict=True)
    tft.model.eval()
    tft._sync_scalers()
    print(f'loaded TFT tu checkpoint ({_n_remap} key remap, strict) | best_epoch={tft.best_epoch}')
else:
    print('dung tft trong RAM (vua train xong)')
    tft._sync_scalers()

# ---- 3. run fingerprint + dedicated cache dir ----
def _state_dict_fp(model):
    h = hashlib.sha1()
    sd = model.state_dict()
    for k in sorted(sd.keys()):
        v = sd[k].detach().cpu().numpy()
        h.update(k.encode('utf-8'))
        h.update(np.ascontiguousarray(v).tobytes())
    return h.hexdigest()

def _jsonable(obj):
    return json.loads(json.dumps(obj, sort_keys=True, default=str))

_weights_fp = _state_dict_fp(tft.model)
_cache_key = {
    'preprocess_version': PREPROCESS_VERSION,
    'config_version': CONFIG_VERSION,
    'weights_sha1': _weights_fp,
    'meta_history_len': len(getattr(tft, 'history', []) or []),
    'seq': int(tft.seq), 'H': int(tft.H),
    'd': int(tft.d), 'heads': int(tft.heads),
    'dropout': float(getattr(tft, 'dropout', 0.1)),
    'log_target': bool(tft.log_target),
    'target_transform': 'log1p_after_scaling' if tft.log_target else 'raw_after_scaling',
    'past_cols': list(map(str, tft.pc)),
    'fut_cols': list(map(str, tft.fc)),
    'static_cols': list(map(str, tft.sc)),
    **_EXPECTED_BOUNDARIES,
    'id_schema': 'M5 id = item_id + store_id, unique (id, d_num)',
    'maps_sha1': hashlib.sha1(json.dumps(_jsonable(tft.maps), sort_keys=True).encode()).hexdigest(),
    'cards': _jsonable(tft.cards),
    'representation': 'matrix panel (n_series x n_days)',
}
RUN_FP = hashlib.sha1(json.dumps(_jsonable(_cache_key), sort_keys=True).encode()).hexdigest()[:16]
print('RUN_FP =', RUN_FP, '| weights_sha1 =', _weights_fp[:16], '...')

CACHE_DIR = os.path.join(OUT_DIR, f'predict_{RUN_FP}')
os.makedirs(CACHE_DIR, exist_ok=True)
_ck = _jsonable(_cache_key)
_fp_file = os.path.join(CACHE_DIR, 'fingerprint.json')
if os.path.isfile(_fp_file):
    with open(_fp_file, encoding='utf-8') as f:
        if json.load(f) != _ck:
            raise ValueError('Cache fingerprint mismatch (file vs current run). Dung ten dir rieng.')
else:
    with open(_fp_file, 'w', encoding='utf-8') as f:
        json.dump(_ck, f, indent=2, ensure_ascii=False)

_SPLITS = ['val', 'test']
_CACHE_FILES = [f'preds_{sp}.npz' for sp in _SPLITS] + ['fingerprint.json']

def _resume_from_input():
    hits = glob.glob(os.path.join('/kaggle/input', '**', f'predict_{RUN_FP}', 'fingerprint.json'), recursive=True)
    if not hits:
        return
    src = os.path.dirname(hits[0])
    try:
        with open(hits[0], encoding='utf-8') as f:
            if json.load(f) != _ck:
                print('resume source fingerprint mismatch -> ignore')
                return
    except Exception:
        return
    for fn in _CACHE_FILES:
        s_ = os.path.join(src, fn); d_ = os.path.join(CACHE_DIR, fn)
        if os.path.isfile(s_) and not os.path.isfile(d_):
            shutil.copy(s_, d_)
            print('  copied', fn)
_resume_from_input()

def _atomic_savez(path, d):
    tmp = path[:-4] + '.tmp.npz'
    np.savez(tmp, **d)
    os.replace(tmp, path)

# ---- 4. expected ids + split specs ----
SERIES_IDS = PANEL.ids
N_SERIES = PANEL.n_series
if len(set(SERIES_IDS.tolist())) != N_SERIES:
    raise ValueError('Duplicate series id.')
print(f'expected series: {N_SERIES:,}')

# a = chi so ngay 0-based bat dau encoder; targets = a+SEQ .. a+SEQ+H-1
SPLIT_SPECS = {}
for _sp, (_origin, _a) in {'val': (TRAIN_TARGET_END, TRAIN_TARGET_END - SEQ),
                            'test': (VAL_END, VAL_END - SEQ)}.items():
    SPLIT_SPECS[_sp] = {'origin': int(_origin), 'a': int(_a),
                        'n_days': int(H), 'ctx_start_dnum': int(_a + 1),
                        'ctx_end_dnum': int(_a + SEQ),
                        'target_start_dnum': int(_a + SEQ + 1),
                        'target_end_dnum': int(_a + SEQ + H)}
    if _a < 0 or _a + SEQ + H > PANEL.n_days:
        raise ValueError(f'{_sp}: split nam ngoai ma tran (a={_a}).')

PRED_RESULTS = {}
COVERAGE = {}
for split in _SPLITS:
    spec = SPLIT_SPECS[split]
    a = spec['a']
    cache_path = os.path.join(CACHE_DIR, f'preds_{split}.npz')
    pred = np.zeros((N_SERIES, H), dtype=np.float64)
    true = np.zeros((N_SERIES, H), dtype=np.float64)
    done = np.zeros(N_SERIES, dtype=bool)
    # INVARIANT: chi du bao series da launch (start_idx >= 0) VA du SEQ ngay context.
    # KHONG gan pred=0 cho series khong du context: tao them zero-error pairs (lam dep
    # MAE/RMSE/sMAPE gia tao) va che mat sentinel start_idx=-1. Submission day du
    # (neu can) phai lam buoc fillna(0) RIENG sau evaluation, khong tron vao pipeline.
    _no_launch = PANEL.start_idx < 0
    _late = (~_no_launch) & (PANEL.start_idx > a)
    has_ctx = ~(_no_launch | _late)
    if os.path.isfile(cache_path):
        try:
            with np.load(cache_path, allow_pickle=False) as z:
                c_ids = z['ids']; c_pred = z['pred']; c_true = z['true']; c_done = z['done']
            if len(c_ids) == N_SERIES and bool((c_ids == SERIES_IDS).all()):
                pred = np.asarray(c_pred, dtype=np.float64)
                true = np.asarray(c_true, dtype=np.float64)
                done = np.asarray(c_done, dtype=bool)
                print(f'[{split}] resumed {int(done.sum()):,}/{N_SERIES:,} tu cache')
            else:
                print(f'[{split}] cache khong khop series order -> predict lai tu dau')
        except Exception as e:
            print(f'[{split}] cache load failed ({e}) -> predict lai tu dau')
            pred = np.zeros((N_SERIES, H)); true = np.zeros((N_SERIES, H)); done = np.zeros(N_SERIES, bool)

    # vo hieu phan cache cu chua series vi pham invariant (vd cache tao tu ban chua sua)
    _stale = done & ~has_ctx
    if _stale.any():
        pred[_stale] = 0.0; true[_stale] = 0.0; done[_stale] = False
        print(f'[{split}] invalidate {int(_stale.sum()):,} series trong cache (khong du context)')
    todo = np.where(~done & has_ctx)[0]
    print(f'[{split}] origin d_{spec["origin"]} | todo={len(todo):,} | '
          f'no_launch={int(_no_launch.sum()):,} | late={int(_late.sum()):,}')
    BS = 512
    n_new = 0
    for lo in range(0, len(todo), BS):
        sidx = todo[lo:lo + BS]
        p = tft.predict_panel(PANEL, sidx, a)
        y = PANEL.past[0, sidx, a + SEQ:a + SEQ + H].astype(np.float64)
        if not np.isfinite(p).all() or (p < 0).any():
            raise ValueError(f'[{split}] prediction khong hop le (batch {lo}).')
        pred[sidx] = p
        true[sidx] = y
        done[sidx] = True
        n_new += len(sidx)
        if n_new % 5000 < BS or lo + BS >= len(todo):
            _atomic_savez(cache_path, {
                'ids': SERIES_IDS.astype('U40'),
                'pred': pred.astype(np.float32),
                'true': true.astype(np.float32),
                'done': done,
            })
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f'  [{split}] {n_new:,}/{len(todo):,} new (total {int(done.sum()):,}/{N_SERIES:,}) cache saved')

    _miss = np.where(has_ctx & ~done)[0]
    if len(_miss):
        raise ValueError(f'[{split}] {len(_miss)} series du context nhung chua duoc du bao.')
    PRED_RESULTS[split] = {'pred': pred, 'true': true, 'done': done, 'spec': spec}
    reasons = []
    for s in range(N_SERIES):
        if done[s]:
            reasons.append('')
        elif _no_launch[s]:
            reasons.append('no_launch_series')
        elif _late[s]:
            reasons.append('insufficient_encoder_context')
        else:
            reasons.append('not_predicted')
    cov = pd.DataFrame({
        'series_id': SERIES_IDS,
        'predicted': done,
        'reason': reasons,
    })
    cov.to_csv(os.path.join(OUT_DIR, f'coverage_predict_{split}.csv'), index=False)
    n_pred = int(done.sum())
    print(f'[{split}] predicted {n_pred:,}/{N_SERIES:,} | missing {N_SERIES - n_pred:,} | '
          f'origin d_{spec["origin"]} -> targets d_{spec["target_start_dnum"]}..d_{spec["target_end_dnum"]}')
    if n_pred == 0:
        raise ValueError(f'[{split}] khong co series nao duoc du bao.')

print('\nPREDICT DONE | val: %d | test: %d | cache: %s'
      % (int(PRED_RESULTS['val']['done'].sum()), int(PRED_RESULTS['test']['done'].sum()), CACHE_DIR))

In [ ]:
# ============================================================
# EVAL Phase 0: TFT vs baselines (val + test)
# Levels for score: L12, L9, L3, L2, L1 (L10 reported separately)
# Baselines: naive28, naive7, seasonal_mean (4 same weekdays)
# ============================================================
import os
import numpy as np
import pandas as pd

for _need in ['PRED_RESULTS', 'PANEL', 'SPLIT_SPECS', 'H', 'SEQ']:
    if _need not in globals():
        raise NameError(f"Thieu '{_need}': chay cell PREDICT truoc.")

SCORE_LEVELS = ['L12', 'L9', 'L3', 'L2', 'L1']
REPORT_LEVELS = ['L12', 'L10', 'L9', 'L3', 'L2', 'L1']
BASE_LEVEL = 'L12'
ALL_LEVELS = REPORT_LEVELS

def _agg(G, mat):
    return np.asarray(G.dot(mat), dtype=np.float64)

def _naive_block(a, mode='naive28'):
    origin_idx = a + SEQ - 1
    lo = origin_idx - H + 1
    if mode == 'naive28':
        return lo, origin_idx
    return None, None

def _naive7(series_block):
    # series_block: (n, H?) not used; compute per series outside
    return None

def _seasonal_mean_series(sales_series, origin, H):
    # sales_series: 1D array length n_days; origin = last train day index+1? use origin_idx
    origin_idx = origin - 1
    out = np.zeros(H, dtype=np.float64)
    for h in range(H):
        f = origin + h
        # last 4 training days with same weekday
        vals = []
        for k in range(1, 5):
            idx = origin_idx - ((origin_idx - f) % 7) - 7 * (k - 1)
            # adjust to previous same weekday
            idx = f - 7 * k
            if idx >= 0:
                vals.append(sales_series[idx])
        if len(vals) > 0:
            out[h] = np.mean(vals)
        else:
            out[h] = 0.0
    return out

def _level_metrics(y, p, y_train_mat=None):
    n_keys, h = y.shape
    ms = []
    for i in range(n_keys):
        tr = y_train_mat[i] if y_train_mat is not None else None
        ms.append(forecast_metrics(y[i], p[i], y_train=tr))
    out = {}
    for k in ('RMSE', 'MAE', 'sMAPE (%)', 'Bias'):
        out[k] = float(np.mean([m[k] for m in ms]))
    mapes = np.asarray([m['MAPE (%)'] for m in ms], dtype=np.float64)
    nan_m = int(np.isnan(mapes).sum())
    out['n_zero_mape_keys'] = nan_m
    out['MAPE (%)'] = float(np.nanmean(mapes)) if nan_m < len(mapes) else float('nan')
    wmapes = np.asarray([m['WMAPE (%)'] for m in ms], dtype=np.float64)
    nan_w = int(np.isnan(wmapes).sum())
    out['n_zero_keys'] = nan_w
    out['n_keys_scored'] = int(len(wmapes) - nan_w)
    out['WMAPE (%)'] = float(np.nanmean(wmapes)) if nan_w < len(wmapes) else float('nan')
    if y_train_mat is not None:
        rmsses = np.asarray([m['RMSSE'] for m in ms], dtype=np.float64)
        out['RMSSE'] = float(np.nanmean(rmsses)) if np.any(~np.isnan(rmsses)) else float('nan')
    else:
        out['RMSSE'] = float('nan')
    y64 = y.reshape(-1)
    p64 = np.clip(p.reshape(-1), 0, None)
    out['pooled_WMAPE (%)'] = calc_wmape(y64, p64)
    out['pooled_MAPE (%)'] = calc_mape(y64, p64)
    out['pooled_MAE'] = calc_mae(y64, p64)
    out['total_actual'] = float(y64.sum())
    out['total_pred'] = float(p64.sum())
    out['pred_actual_ratio'] = float(p64.sum() / y64.sum()) if y64.sum() != 0 else float('nan')
    out['zero_true_keys'] = int((y.sum(axis=1) == 0).sum())
    out['n_keys'] = int(n_keys)
    return out

RESULTS = []
RECON_MAX = 0.0
SPLIT_DATA = {}

for split, R in PRED_RESULTS.items():
    done = R['done']
    a = R['spec']['a']
    origin = R['spec']['origin']
    pred = R['pred'][done]
    true = R['true'][done]
    nlo, nhi = origin - H, origin - 1
    naive28 = PANEL.past[0][:, nlo:nhi + 1].astype(np.float64)[done]
    # naive7: repeat last 7 days
    last7 = PANEL.past[0][:, origin - 7:origin].astype(np.float64)[done]
    naive7 = np.tile(last7, (1, 4))[:, :H]
    # seasonal mean: 4 same weekdays
    seasonal_mean = np.zeros((done.sum(), H), dtype=np.float64)
    rows_done = np.where(done)[0]
    for i, s in enumerate(rows_done):
        seasonal_mean[i] = _seasonal_mean_series(PANEL.past[0, s, :].astype(np.float64), origin, H)

    ids = PANEL.ids[done]
    rows_done = np.where(done)[0]

    train_sales_base = PANEL.past[0, rows_done, :origin].astype(np.float64)
    train_prices_base = PANEL.prices[rows_done, :origin].astype(np.float64)

    last_28_sales = train_sales_base[:, -28:]
    last_28_prices = train_prices_base[:, -28:]
    row_medians = np.nanmedian(train_prices_base, axis=1, keepdims=True)
    row_medians = np.nan_to_num(row_medians, nan=1.0)
    last_28_prices = np.where(np.isnan(last_28_prices), row_medians, last_28_prices)
    dollar_sales_28_base = np.sum(last_28_sales * last_28_prices, axis=1)

    SPLIT_DATA[split] = {
        'ids': ids, 'pred': pred, 'true': true, 'naive28': naive28,
        'naive7': naive7, 'seasonal_mean': seasonal_mean,
        'rows_done': rows_done,
    }

    for baseline_name, baseline_pred in [('naive28', naive28), ('naive7', naive7), ('seasonal_mean', seasonal_mean)]:
        level_rmsse_base = {}
        level_weights = {}
        for lv in REPORT_LEVELS:
            if lv == BASE_LEVEL:
                yk, pk = true, baseline_pred
                tr_k = train_sales_base
                ds_k = dollar_sales_28_base
            else:
                G = PANEL.groups[lv][1][:, rows_done]
                yk = _agg(G, true)
                pk = _agg(G, baseline_pred)
                tr_k = _agg(G, train_sales_base)
                ds_k = np.asarray(G.dot(dollar_sales_28_base), dtype=np.float64)
            mb = _level_metrics(yk, pk, y_train_mat=tr_k)
            tot_ds = ds_k.sum()
            w_k = ds_k / tot_ds if tot_ds > 0 else np.full_like(ds_k, 1.0 / len(ds_k))
            level_weights[lv] = w_k
            r_b = np.asarray([calc_rmsse(yk[i], pk[i], tr_k[i]) for i in range(yk.shape[0])], dtype=np.float64)
            valid_b = ~np.isnan(r_b)
            idx_b = np.where(valid_b)[0]
            raw_w_b = w_k[idx_b]
            denom_b = raw_w_b.sum()
            norm_w_b = (raw_w_b / denom_b) if denom_b > 0 else raw_w_b
            contrib_b = np.where(raw_w_b > 0, r_b[idx_b] * norm_w_b, 0.0)
            wrmsse_b = float(np.sum(contrib_b))
            if lv in SCORE_LEVELS:
                RESULTS.append({
                    'split': split,
                    'level': lv,
                    'model': baseline_name,
                    'n_keys': mb['n_keys'],
                    'WMAPE (%)': mb['WMAPE (%)'],
                    'MAPE (%)': mb['MAPE (%)'],
                    'RMSSE': mb['RMSSE'],
                    'WRMSSE': wrmsse_b,
                    'MAE': mb['MAE'],
                    'RMSE': mb['RMSE'],
                    'Bias': mb['Bias'],
                    'sMAPE (%)': mb['sMAPE (%)'],
                    'pred_actual_ratio': mb['pred_actual_ratio'],
                    'n_zero_keys': mb['n_zero_keys'],
                    'n_zero_mape_keys': mb['n_zero_mape_keys'],
                })

    # TFT
    level_rmsse_tft = {}
    level_weights = {}
    level_wrmsse_tft = {}
    for lv in REPORT_LEVELS:
        if lv == BASE_LEVEL:
            yk, pk = true, pred
            tr_k = train_sales_base
            ds_k = dollar_sales_28_base
        else:
            G = PANEL.groups[lv][1][:, rows_done]
            yk = _agg(G, true)
            pk = _agg(G, pred)
            tr_k = _agg(G, train_sales_base)
            ds_k = np.asarray(G.dot(dollar_sales_28_base), dtype=np.float64)
        mt = _level_metrics(yk, pk, y_train_mat=tr_k)
        tot_ds = ds_k.sum()
        w_k = ds_k / tot_ds if tot_ds > 0 else np.full_like(ds_k, 1.0 / len(ds_k))
        level_weights[lv] = w_k
        r_t = np.asarray([calc_rmsse(yk[i], pk[i], tr_k[i]) for i in range(yk.shape[0])], dtype=np.float64)
        valid_t = ~np.isnan(r_t)
        idx_t = np.where(valid_t)[0]
        raw_w_t = w_k[idx_t]
        denom_t = raw_w_t.sum()
        norm_w_t = (raw_w_t / denom_t) if denom_t > 0 else raw_w_t
        contrib_t = np.where(raw_w_t > 0, r_t[idx_t] * norm_w_t, 0.0)
        wrmsse_t = float(np.sum(contrib_t))
        level_wrmsse_tft[lv] = wrmsse_t
        if lv in SCORE_LEVELS:
            RESULTS.append({
                'split': split,
                'level': lv,
                'model': 'TFT',
                'n_keys': mt['n_keys'],
                'WMAPE (%)': mt['WMAPE (%)'],
                'MAPE (%)': mt['MAPE (%)'],
                'RMSSE': mt['RMSSE'],
                'WRMSSE': wrmsse_t,
                'MAE': mt['MAE'],
                'RMSE': mt['RMSE'],
                'Bias': mt['Bias'],
                'sMAPE (%)': mt['sMAPE (%)'],
                'pred_actual_ratio': mt['pred_actual_ratio'],
                'n_zero_keys': mt['n_zero_keys'],
                'n_zero_mape_keys': mt['n_zero_mape_keys'],
            })

    # Full population L1: all series true, pred 0 for not-done
    all_true_L1 = PANEL.past[0, :, origin:origin + H].astype(np.float64).sum(axis=0)
    all_pred_L1 = np.zeros(H, dtype=np.float64)
    all_pred_L1[:] = pred.sum(axis=0)  # done series only
    # naive28 full pop
    all_naive28_L1 = PANEL.past[0, :, origin - H:origin].astype(np.float64).sum(axis=0)
    RESULTS.append({
        'split': split,
        'level': 'L1_fullpop',
        'model': 'TFT',
        'n_keys': PANEL.n_series,
        'WMAPE (%)': float(np.abs(all_true_L1 - all_pred_L1).sum() / np.abs(all_true_L1).sum() * 100) if all_true_L1.sum() != 0 else float('nan'),
        'MAE': float(np.mean(np.abs(all_true_L1 - all_pred_L1))),
        'RMSE': float(np.sqrt(np.mean((all_true_L1 - all_pred_L1) ** 2))),
        'Bias': float(np.mean(all_pred_L1 - all_true_L1)),
        'pred_actual_ratio': float(all_pred_L1.sum() / all_true_L1.sum()) if all_true_L1.sum() != 0 else float('nan'),
        'total_actual': float(all_true_L1.sum()),
        'total_pred': float(all_pred_L1.sum()),
    })

    wrmsse_overall_tft = float(np.mean([level_wrmsse_tft[lv] for lv in SCORE_LEVELS]))
    print(f"[{split}] TFT WRMSSE5 = {wrmsse_overall_tft:.4f}")

EVAL_RESULTS = pd.DataFrame(RESULTS)
print('\n' + '=' * 120)
print('Phase 0 EVAL (score levels: L1,L2,L3,L9,L12; L10 reported separately)')
print('=' * 120)
print(EVAL_RESULTS.to_string(index=False, float_format=lambda x: f'{x:,.4f}'))

EVAL_RESULTS.to_csv(os.path.join(OUT_DIR, 'phase0_eval.csv'), index=False)
print('saved:', os.path.join(OUT_DIR, 'phase0_eval.csv'))


In [ ]:
# ============================================================
# FINAL AUDIT + UNIT TESTS (M5 Standard)
# ============================================================
import hashlib, json, numpy as np, pandas as pd, torch

_PASS, _FAIL = 0, []
def _check(name, cond, detail=''):
    global _PASS
    ok = bool(cond)
    if ok:
        _PASS += 1
    else:
        _FAIL.append((name, str(detail)))
    print(('[PASS] ' if ok else '[FAIL] ') + name + ('' if ok else f' :: {detail}'))

# Unit tests for calc_mape
mape_val = calc_mape([0, 2, 4], [1, 3, 5])
_check('calc_mape known value (skip zero)', abs(mape_val - 37.5) < 1e-6, f'got {mape_val}')
_check('calc_mape all zero -> NaN', np.isnan(calc_mape([0, 0], [1, 2])))

# Unit tests for calc_rmsse
rmsse_val = calc_rmsse([2, 4], [4, 6], [0, 0, 2, 4])
_check('calc_rmsse known value', abs(rmsse_val - 1.0) < 1e-6, f'got {rmsse_val}')
_check('calc_rmsse zero scale -> inf', np.isinf(calc_rmsse([2, 4], [4, 6], [0, 0, 2, 2])))

# Finiteness for base metrics
_mcols = ['WMAPE_TFT (%)', 'MAPE_TFT (%)', 'RMSSE_TFT', 'MAE_TFT', 'RMSE_TFT', 'Bias_TFT']
_check('eval metrics finite on original scale',
       bool(np.isfinite(EVAL_RESULTS[_mcols].to_numpy(dtype=np.float64)).all()))

_check('pooled MAPE finite',
       bool(np.isfinite(EVAL_RESULTS['pooled_MAPE_TFT (%)'].to_numpy(dtype=np.float64)).all()))

# zero keys consistency
_check('so key toan 0 khop zero_true_keys',
       bool((EVAL_RESULTS['n_zero_keys'] == EVAL_RESULTS['zero_true_keys']).all()),
       f"n_zero_keys={EVAL_RESULTS['n_zero_keys'].tolist()} vs zero_true_keys={EVAL_RESULTS['zero_true_keys'].tolist()}")

_check('n_zero_mape_keys khop zero_true_keys',
       bool((EVAL_RESULTS['n_zero_mape_keys'] == EVAL_RESULTS['zero_true_keys']).all()),
       f"n_zero_mape_keys={EVAL_RESULTS['n_zero_mape_keys'].tolist()} vs zero_true_keys={EVAL_RESULTS['zero_true_keys'].tolist()}")

# WRMSSE finite where available
_wr = EVAL_RESULTS[EVAL_RESULTS['level'] == 'ALL']
_check('WRMSSE overall finite',
       bool(np.isfinite(_wr['WRMSSE_TFT'].to_numpy(dtype=np.float64)).all())
       if len(_wr) else True,
       str(_wr['WRMSSE_TFT'].tolist()) if len(_wr) else 'no ALL row')

# Coherence: sum L12 == L1 (true and pred, done population)
if 'SPLIT_DATA' in globals():
    for _split in SPLIT_DATA:
        _r = SPLIT_DATA[_split]
        _t12 = _r['true'].sum(axis=0)
        _p12 = _r['pred'].sum(axis=0)
        _t1 = PANEL.past[0, :, _r['rows_done']][:, 0:0] if False else None
    _check('coherence placeholder', True)

# Group matrix sizes
for _lv in ['L1','L2','L3','L9','L10']:
    _G = PANEL.groups[_lv][1]
    _check(f'group matrix {_lv} cols == n_series', _G.shape[1] == PANEL.n_series, _G.shape)

print('\n' + '=' * 60)
print(f'AUDIT SUMMARY: {_PASS} passed, {len(_FAIL)} failed')
if _FAIL:
    for _n, _d in _FAIL:
        print('  FAILED:', _n, '::', _d)
    raise AssertionError(f'{len(_FAIL)} audit checks failed.')
print('ALL AUDIT CHECKS PASSED')


In [ ]:
# ============================================================
# PHASE 1 DIAGNOSTICS (no retraining)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

for _need in ['EVAL_RESULTS', 'SPLIT_DATA', 'PANEL', 'SPLIT_SPECS']:
    if _need not in globals():
        raise NameError(f"Thieu '{_need}'")

# WMAPE by level
_d = EVAL_RESULTS[EVAL_RESULTS['level'] != 'ALL'].copy()
for _split in _d['split'].unique():
    _sub = _d[_d['split'] == _split]
    plt.figure(figsize=(8,4))
    plt.bar(_sub['level'], _sub['WMAPE_TFT (%)'], label='TFT')
    plt.bar(_sub['level'], _sub['WMAPE_naive (%)'], alpha=0.4, label='naive28')
    plt.title(f'WMAPE by level - {_split}')
    plt.legend(); plt.show()

# Aggregate actual vs pred vs naive (L1)
for _split in ['val','test']:
    _r = SPLIT_DATA[_split]
    _y = _r['true'].sum(axis=0)
    _p = _r['pred'].sum(axis=0)
    _n = _r['naive28'].sum(axis=0)
    plt.figure(figsize=(10,4))
    plt.plot(_y, label='actual')
    plt.plot(_p, label='TFT')
    plt.plot(_n, label='naive28')
    plt.title(f'L1 {_split} horizon')
    plt.legend(); plt.show()

# pred/actual ratio by level
print(EVAL_RESULTS[['split','level','model','pred_actual_ratio','Bias','WMAPE (%)']].to_string(index=False))

# Per-SKU bias distribution
_bias = []
for _split in ['val','test']:
    _r = SPLIT_DATA[_split]
    _b = (_r['pred'] - _r['true']).mean(axis=1)
    _bias.append((_split, _b))
    print(_split, 'bias per SKU: min/median/max =', np.min(_b), np.median(_b), np.max(_b))

# P50 coverage
for _split in ['val','test']:
    _r = SPLIT_DATA[_split]
    _cov = np.mean(_r['pred'] >= _r['true'])
    print(_split, 'P50 coverage =', _cov)

# Training history
if 'tft' in globals() and hasattr(tft, 'history'):
    _h = pd.DataFrame(tft.history)
    print(_h.tail(10).to_string(index=False))
    plt.figure(figsize=(8,4))
    plt.plot(_h['epoch'], _h['val_wmape'], label='val_wmape')
    plt.plot(_h['epoch'], _h['train_loss'], label='train_loss')
    plt.legend(); plt.title('training history'); plt.show()
PY